# External audio audit + zero-shot transfer: UAV Propeller Anomaly Audio → TII audio

**Question.** Can an audio classifier trained **only** on the external *UAV Propeller Anomaly Audio Data Set*
(Katta et al., ICASSP 2022, `github.com/tiiuae/UAV-Propeller-Anomaly-Audio-Dataset`) recognize the number of broken
propellers in TII audio (same lab, reportedly the same drone)?

- **Part 1 — audit (sections 1–3).** Both datasets are audited, then their compatibility: format, sample rate,
  channel, gain, and possible overlap.
- **Part 2 — zero-shot transfer (sections 4–7).** Models are trained on external audio only and tested on all 95 TII
  missions. They are compared with an in-domain TII-from-scratch reference built with the **identical** front end, and
  with a no-training loudness baseline (RMS-CENTROID).

**What this tests.** Compatibility only. It does **not** test whether pretraining on the external data improves final
results. That is a later notebook, run only if the gate in section 7 passes.

**Out of scope.** IMU/telemetry and any cross-modal work. Only TII `AudioBuffer` files are downloaded and read.

**Self-sufficient.** The notebook downloads both datasets from GitHub, installs what it needs, and contains every
helper it uses (copied from the revised audio notebook `TII_Audio_Only_Analysis_Mamba.ipynb` and from
`version 28/minor-project-v28.ipynb`, audio arm A1). Run it on Kaggle with a **GPU** and **internet on**. All results
go to `/kaggle/working/`.

**Pre-registered (fixed before any result is seen).**
- Headline arm: **EXT_LEVEL + NORMCAL + MEAN5 + assumed rate (16 kHz)**. Every other arm is a diagnostic: all are
  reported, and none is picked as "best" on TII.
- Tiers and gate: section 7.

**Hooks.** Every hook prints PASS/FAIL (soft hooks print FLAG). A failed **hard** hook stops the notebook before any
model is trained. The hard hooks cover external counts and formats, TII counts, split integrity, and real `mamba_ssm`.

**One deviation from v28, stated up front.** v28 trains 20 fixed epochs and keeps the epoch with the best validation
macro-F1. This notebook keeps the 20 fixed epochs but selects the epoch with the **lowest validation loss**, as the
specification requires for the external models. The same rule is used for every model, so EXT and TII-scratch are
trained identically.

## 0. Setup
- **0a** environment, paths, run flags (`QUICK_MODE`).
- **0b** dependencies and real `mamba-ssm` (no fallback model).
- **0c** every constant, in one cell.
- **0d** seeding/determinism, hooks, figure helpers.
- **0e** downloads: the external dataset (`git clone --depth 1`) and the TII AudioBuffer files only (sparse,
  blob-less checkout, falling back to a full shallow clone if that fails).

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, RUN FLAGS
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import copy
import csv
import gc
import hashlib
import json
import math
import random
import re
import shutil
import subprocess
import sys
import time
import wave
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

T_NOTEBOOK_START = time.time()

# ---- run flag ----
QUICK_MODE = False          # True = smoke test: seed 42 only and the LEVEL feature set only

IS_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "audio_zeroshot_outputs"
# Raw git clones are INPUTS, not results. On Kaggle they go to local scratch (/tmp) so that "Save Version" does not have
# to persist several GB of raw data. Every derived file (caches, CSVs, probabilities, checkpoints, figures) is in OUT.
KEEP_RAW_IN_WORKING = False
RAW_DIR = Path("/tmp/audio_zeroshot_raw") if (IS_KAGGLE and not KEEP_RAW_IN_WORKING) else OUT / "raw"
FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "audio_zeroshot_cache"
TII_CACHE = CACHE_DIR / "tii"
PROBS_DIR = OUT / "audio_zeroshot_probs"
CKPT_DIR = OUT / "audio_zeroshot_checkpoints"
RUNS_CSV = OUT / "audio_zeroshot_runs.csv"
EVAL_CSV = OUT / "audio_zeroshot_eval.csv"
for _d in (OUT, RAW_DIR, FIG_DIR, TII_CACHE, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'local'} | outputs -> {OUT} | raw clones -> {RAW_DIR}")
print(f"QUICK_MODE = {QUICK_MODE}")

In [ ]:
# ==============================================================================
# 0b. DEPENDENCIES + REAL mamba-ssm (installs only what is missing; there is NO fallback model)
# ==============================================================================
import importlib
import importlib.util


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


for _mod, _pkg in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
                   ("sklearn", "scikit-learn"), ("torch", "torch")]:
    if importlib.util.find_spec(_mod) is None and not _pip([_pkg]):
        raise RuntimeError(f"could not install {_pkg}: check that internet is ON")
importlib.invalidate_caches()
if shutil.which("git") is None:
    raise RuntimeError("git is not available: it is needed to download both datasets")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: Kaggle -> Settings -> Accelerator -> GPU, then "
                       "re-run. There is deliberately no CPU fallback.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU is "
                           "attached, restart the session and re-run. No fallback model is used.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here (all fixed before any result is seen)
# ==============================================================================
SEEDS = [42] if QUICK_MODE else [42, 123]
FEATURE_SETS = ["LEVEL"] if QUICK_MODE else ["LEVEL", "SHAPE"]
N_CLASSES = 5
CLASS_NAMES = ["0 normal", "1 broken", "2 broken", "3 broken", "4 broken"]

# ---- external: UAV Propeller Anomaly Audio Data Set (Katta et al., ICASSP 2022) ----
EXT_REPO_URL = "https://github.com/tiiuae/UAV-Propeller-Anomaly-Audio-Dataset.git"
EXT_CLASS_DIRS = {"normal": 0, "1_broken": 1, "2_broken": 2, "3_broken": 3, "4_broken": 4}   # 3 and 4 kept separate
MISSION_TYPES = ["ccw_f8", "cw_f8", "ccw_square", "cw_square"]
EXT_RATE = 16000
EXPECTED_EXT = {                                  # facts from the earlier audit; re-verified in section 1
    "n_files": 293, "class_counts": [78, 72, 71, 48, 24], "channels": 1, "rate": 16000, "sample_width_bytes": 2,
    "dur_min_s": 37.6, "dur_max_s": 108.7, "dur_median_s": 60.6, "total_h": 5.20, "total_h_tol": 0.05,
    "max_abs_dc": 0.6, "max_zero_frac": 0.008,
    "lead_median_s": 1.0, "lead_max_s": 6.0, "tail_median_s": 2.0, "tail_max_s": 9.0,
    "steady_rms_class": [75, 168, 415, 454, 472], "rms_rel_tol": 0.10,
    "steady_rms_config": {"0001": 107, "0100": 116, "1000": 279, "0010": 456,
                          "0101": 155, "1001": 314, "1100": 406, "0011": 438, "0110": 457, "1010": 475},
    "frac_2b_quieter_than_median_1b": 0.17, "p_4b_louder_than_3b": 0.75,
}

# ---- TII: UAV-Realistic-Fault-Dataset (AudioBuffer files only) ----
TII_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
TII_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}   # corrupt recording_started_time
EXPECTED_TII_N = 95
EXPECTED_TII_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
N_SLOTS, DROP_SLOT, FRAME = 6, 5, 256           # 1536 = 256 frames x 6 interleaved slots; slot index 5 is all zero
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 channels
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
ASSUMED_TII_RATE = 16000                        # per-channel rate: NOT in the data. Hypothesis only (16 ms per buffer)

# ---- shared front end (identical for both datasets; one channel per sample) ----
N_CACHE_BINS = 65                               # cached rFFT bins 0..64 of the 256-point frame
N_FEAT = 16
TARGET_HZ = 62.5 * np.arange(1, N_FEAT + 1)     # = external bins 1..16 at 16 kHz (62.5..1000 Hz)
WINDOW_FRAMES, STRIDE_FRAMES, TOKEN_POOL = 128, 64, 2
N_TOKENS = WINDOW_FRAMES // TOKEN_POOL          # 64 tokens x 16 features per window
SHAPE_EPS = 1e-6
LOGPOW_EPS = 1e-6
TRIM_REL = 0.5                                  # trim: keep first..last 1-s block with RMS >= 0.5 x steady reference
RMS_BLOCK_S = 1.0


def block_frames(rate):
    """Frames per ~1-second block at `rate` (62 at 16 kHz = 0.992 s)."""
    return max(1, int(round(RMS_BLOCK_S * rate / FRAME)))


# ---- compatibility audit ----
RATE_CANDIDATES = [8000, 10000, 12000, 14000, 16000, 18000, 20000, 24000, 32000]
SCAN_GRID_HZ = np.arange(100.0, 2000.0 + 1e-9, 25.0)
RATE_MARGIN = 0.05                              # best rate must beat the second best by >= this (top-3-channel mean r)
RATE_TOP_CHANNELS = 3
RATE_VARIANT_MIN_REL = 0.05                     # RATE_EST becomes a variant only if it differs from 16 kHz by > 5%
ENVELOPE_POINTS = 100
OVERLAP_FLAG = 0.98

# ---- model + training: version 28/minor-project-v28.ipynb, audio arm A1 (config cell, UnimodalMamba, train loop) ----
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
V28_CHECKPOINT_RULE = "20 fixed epochs, no early stopping; keep the epoch with the highest validation macro-F1"
CHECKPOINT_RULE = "20 fixed epochs (as v28); keep the epoch with the LOWEST class-weighted validation loss"

# ---- validation splits ----
EXT_VAL_FRACTION = 0.15                         # external: whole held-out groups until >= 15% of each class's files
INNER_VAL_FRACTION = 0.15                       # TII scratch: per class ~15% (min 1) of training missions
INNER_VAL_SEED = 2026                           # the v27/v28 fallback inner-validation rule (seed [2026, fold])

# ---- evaluation (pre-registered) ----
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
HEADLINE = {"model": "EXT_LEVEL", "gain": "NORMCAL", "channels": "MEAN5", "rate": ASSUMED_TII_RATE}
CHANNEL_SPECS = [f"c{c + 1}" for c in range(N_CH)] + ["MEAN5"]
TIER_LB_NOT_COMPATIBLE = 0.30
TIER_MARGIN = 0.05
CHANCE_MACRO_F1 = 0.20
CONTEXT_LINES = [
    "Revised audio notebook, A1 (5 channels x 16 bins, day-out): window macro-F1 0.807 [0.754, 0.859], "
    "mission macro-F1 0.845 [0.772, 0.915]",
    "v28 A1 (day-out): window macro-F1 0.741 [0.689, 0.791]",
]

print(f"Seeds {SEEDS} | feature sets {FEATURE_SETS} | frame {FRAME} | features: bins at {TARGET_HZ[0]:.1f}-"
      f"{TARGET_HZ[-1]:.1f} Hz | windows {WINDOW_FRAMES} frames, stride {STRIDE_FRAMES}, {N_TOKENS} tokens x {N_FEAT}")
print(f"Headline arm (pre-registered): {HEADLINE}")

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, HOOKS, SHELL + FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)   # warn_only: mamba-ssm's CUDA kernels are outside the registry
set_all_seeds(SEEDS[0])


def loud(msg):
    print("🚨" * 50)
    for line in str(msg).split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


HOOKS = []


def hook(name, ok, msg, hard=False):
    """Hard hooks stop the notebook on failure (before training). Soft hooks only FLAG a deviation."""
    ok = bool(ok)
    HOOKS.append({"hook": name, "ok": ok, "hard": hard, "msg": msg})
    print(f"[HOOK {'HARD' if hard else 'soft'}] {'PASS' if ok else ('FAIL' if hard else 'FLAG')} — {name}: {msg}")
    if hard and not ok:
        loud(f"HARD HOOK FAILED: {name}\n{msg}\nStopping before any model is trained.")
        raise RuntimeError(f"hard hook failed: {name}")
    return ok


def run_cmd(args, cwd=None):
    print("$ " + " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-3000:])
    return r.returncode


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]


def config_tag(d):
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CH_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
EXT_COLOR = "#1f1f1f"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.spines.top": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10,
                     "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FIGURES = []
RATE_CAVEAT = ("TII Hz values assume a per-channel rate that is NOT in the data (16 kHz unless stated); the external "
               "rate (16 kHz) is known.")


def save_fig(fig, name, caption):
    """Every figure is saved as PNG with a caption that states its assumptions."""
    assert caption, f"{name}: every figure needs a caption"
    fig.text(0.5, -0.02, caption, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append(name)
    print(f"  saved {path}")

In [ ]:
# ==============================================================================
# 0e-1. DOWNLOAD THE EXTERNAL DATASET (git clone --depth 1, ~574 MB)
# ==============================================================================
EXT_REPO = RAW_DIR / "UAV-Propeller-Anomaly-Audio-Dataset"
EXT_ROOT = EXT_REPO / "Dataset"


def _ext_wavs():
    return sorted(EXT_ROOT.rglob("*.wav")) if EXT_ROOT.is_dir() else []


if not _ext_wavs():
    if EXT_REPO.exists():
        shutil.rmtree(EXT_REPO)
    _rc = run_cmd(["git", "clone", "--depth", "1", EXT_REPO_URL, EXT_REPO])
    if _rc != 0 or not _ext_wavs():
        loud(f"The external dataset could not be downloaded from {EXT_REPO_URL} (git exit code {_rc}).\n"
             "Check that internet is ON. Stopping: nothing is guessed.")
        raise RuntimeError("external dataset download failed")
EXT_WAVS = _ext_wavs()
if EXT_WAVS[0].read_bytes()[:40].startswith(b"version https://git-lfs"):
    loud("The external .wav files are Git LFS pointer files, not audio. Stopping.")
    raise RuntimeError("external dataset is LFS pointers")
print(f"External dataset: {EXT_ROOT} | {len(EXT_WAVS)} .wav files | "
      f"{sum(p.stat().st_size for p in EXT_WAVS) / 1e6:.0f} MB")

In [ ]:
# ==============================================================================
# 0e-2. DOWNLOAD ONLY THE TII AudioBuffer FILES (blob-less clone + sparse checkout; fallback: full shallow clone)
# ==============================================================================
TII_REPO = RAW_DIR / "UAV-Realistic-Fault-Dataset"
TII_ROOT = TII_REPO / "Dataset"
TII_PATTERN = "/Dataset/*/*/*AudioBuffer*.jsonl"


def _tii_audio():
    return sorted(TII_ROOT.glob("*/*/*AudioBuffer*.jsonl")) if TII_ROOT.is_dir() else []


TII_DOWNLOAD_METHOD = "already present (earlier session)"
if not _tii_audio():
    if TII_REPO.exists():
        shutil.rmtree(TII_REPO)
    _ok = run_cmd(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", TII_REPO_URL, TII_REPO]) == 0
    if _ok:
        _ok = run_cmd(["git", "-C", TII_REPO, "sparse-checkout", "set", "--no-cone", TII_PATTERN]) == 0
        if not _ok:                                   # older git: no --no-cone flag on `set` (non-cone is the default)
            _ok = (run_cmd(["git", "-C", TII_REPO, "sparse-checkout", "init"]) == 0
                   and run_cmd(["git", "-C", TII_REPO, "sparse-checkout", "set", TII_PATTERN]) == 0)
    if _ok:
        run_cmd(["git", "-C", TII_REPO, "checkout"])
        if not _tii_audio():                          # some git versions need an explicit tree read after --no-checkout
            run_cmd(["git", "-C", TII_REPO, "read-tree", "-mu", "HEAD"])
    if _ok and _tii_audio():
        TII_DOWNLOAD_METHOD = f"blob-less shallow clone + sparse checkout of {TII_PATTERN}"
    else:
        loud("Sparse checkout of the TII AudioBuffer files FAILED.\n"
             "FALLBACK: full shallow clone of the TII repository (~4.4 GB). Only AudioBuffer files will be read.")
        shutil.rmtree(TII_REPO, ignore_errors=True)
        _rc = run_cmd(["git", "clone", "--depth", "1", TII_REPO_URL, TII_REPO])
        TII_DOWNLOAD_METHOD = "FALLBACK: full git clone --depth 1"
        if _rc != 0 or not _tii_audio():
            loud(f"The TII dataset could not be downloaded from {TII_REPO_URL} (git exit code {_rc}). Stopping.")
            raise RuntimeError("TII download failed")
TII_AUDIO_FILES = _tii_audio()
with open(TII_AUDIO_FILES[0], "rb") as _f:
    if _f.read(40).startswith(b"version https://git-lfs"):
        loud("The TII AudioBuffer files are Git LFS pointer files, not data. Stopping.")
        raise RuntimeError("TII files are LFS pointers")
print(f"TII download: {TII_DOWNLOAD_METHOD}")
print(f"TII AudioBuffer files found: {len(TII_AUDIO_FILES)} | "
      f"{sum(p.stat().st_size for p in TII_AUDIO_FILES) / 1e9:.2f} GB")

## 1. External audit
- **1a** — every `.wav` file is read once. The notebook checks its format, computes per-frame spectra (256-sample
  frames, mean removed, rectangular window, rFFT bins 0–64), and records the per-frame RMS.
- **Trimming rule (applied identically to TII in section 2).**
  - Split the recording into ~1-second blocks of frames.
  - Reference = the median block RMS over the middle 50% of the blocks. This is the **steady-flight RMS**.
  - Keep the span from the first to the last block whose RMS is ≥ 0.5 × the reference. Only that span becomes model
    input.
- **1b** — the earlier audit's facts (items 1–5) are re-verified with hooks.
  - **Hard (stop before training):** file and class counts; mono, 16 kHz, 16-bit format.
  - **Soft (FLAG only):** durations, duplicates, clipping, DC, zeros, quiet lead-in and tail, steady RMS per class
    (within 10%) and per configuration.
- Saved: `external_metadata.csv`.

**Labels.** `normal` = 0, `1_broken` = 1, `2_broken` = 2, `3_broken` = 3, `4_broken` = 4. The paper merges 3 and 4,
but the folders keep them separate, so they stay separate here.

**Note.** The paper's 98% accuracy used a random 80/10/10 split of audio chunks, which leaks flights and
configurations. No chunk-level split is used anywhere in this notebook.

In [ ]:
# ==============================================================================
# 1a. EXTERNAL: PARSE PATHS, READ EVERY WAV ONCE, FRAME SPECTRA, STEADY RMS, TRIM
# ==============================================================================
def parse_ext_path(p):
    """Dataset/<class>/[<configuration>]/<mission_type>/<run>/ch1.wav; 3_4_broken has 3_broken / 4_broken below it.
    Deeper or extra levels are discovered, not assumed: the mission type is located by name."""
    rel = p.relative_to(EXT_ROOT).parts
    if rel[0] == "3_4_broken":
        cls_dir, rest = rel[1], rel[2:]
    else:
        cls_dir, rest = rel[0], rel[1:]
    if cls_dir not in EXT_CLASS_DIRS:
        raise ValueError(f"unknown class folder {cls_dir!r} in {p}")
    mt_idx = next((i for i, part in enumerate(rest) if part in MISSION_TYPES), None)
    if mt_idx is None:
        raise ValueError(f"no mission-type folder in {p}")
    pre = rest[:mt_idx]
    config = next((x for x in pre if re.fullmatch(r"[01]{4}", x)), None)
    if config is None:
        config = "0000" if cls_dir == "normal" else "unknown"
    return {"class": EXT_CLASS_DIRS[cls_dir], "class_dir": cls_dir, "configuration": config,
            "mission_type": rest[mt_idx], "run": "/".join(rest[mt_idx + 1:-1]) or "?",
            "extra_levels": "/".join(x for x in pre if x != config), "file_name": rest[-1]}


def frame_spectra(x):
    """x: (n_frames, FRAME). Per-frame mean removed, rectangular window. Returns rFFT power bins 0..64 and frame RMS."""
    x = x - x.mean(axis=1, keepdims=True)
    rms = np.sqrt((x ** 2).mean(axis=1))
    P = (np.abs(np.fft.rfft(x, axis=1)) ** 2)[:, :N_CACHE_BINS]
    return P.astype(np.float32), rms.astype(np.float32)


def block_rms(frame_ms, B):
    nb = len(frame_ms) // B
    return np.sqrt(frame_ms[: nb * B].reshape(nb, B).mean(axis=1))


def trim_span(frame_ms, B):
    """Shared trimming rule (external AND TII). ~1-s blocks of B frames; reference = median block RMS over the middle
    50% of blocks (= steady-flight RMS); keep first..last block with RMS >= TRIM_REL x reference. Frame range [a, b)."""
    br = block_rms(frame_ms, B)
    nb = len(br)
    if nb == 0:
        return None
    mid = br[nb // 4: max(nb // 4 + 1, (3 * nb) // 4)]
    ref = float(np.median(mid))
    keep = np.where(br >= TRIM_REL * ref)[0] if ref > 0 else np.array([], int)
    if len(keep) == 0:
        return None
    first, last = int(keep[0]), int(keep[-1])
    return {"a": first * B, "b": (last + 1) * B, "ref_rms": ref, "lead_blocks": first, "tail_blocks": nb - 1 - last,
            "n_blocks": nb, "block_rms": br}


EXT_DATA, _rows = {}, []
_t0 = time.time()
for i, p in enumerate(EXT_WAVS):
    info = parse_ext_path(p)
    with wave.open(str(p), "rb") as w:
        nch, sw, sr, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
        raw = w.readframes(n)
    fid = f"ext_{i:03d}"
    row = {"file_id": fid, "path": str(p.relative_to(EXT_ROOT)), **info, "n_channels": nch, "sample_rate": sr,
           "sample_width_bytes": sw, "n_samples": n, "duration_s": n / sr, "sha1": hashlib.sha1(raw).hexdigest()}
    if sw != 2:
        row["status"] = f"unsupported sample width {sw} bytes"
        _rows.append(row)
        continue
    x = np.frombuffer(raw, dtype="<i2")
    if nch > 1:
        x = x.reshape(-1, nch)[:, 0]                  # flagged by the format hook; only the first channel is used
    x = x.astype(np.float64)
    row.update({"dc_mean": float(x.mean()), "zero_frac": float((x == 0).mean()), "std": float(x.std()),
                "n_clipped": int(((x >= 32767) | (x <= -32768)).sum())})
    nf = len(x) // FRAME
    P, rms = frame_spectra(x[: nf * FRAME].reshape(nf, FRAME))
    ms = rms.astype(np.float64) ** 2
    B = block_frames(sr)
    span = trim_span(ms, B)
    if span is None:
        row["status"] = "dead or too short"
        _rows.append(row)
        continue
    blk_s = B * FRAME / sr
    row.update({"status": "ok", "n_frames": nf, "steady_rms": span["ref_rms"], "lead_quiet_s": span["lead_blocks"] * blk_s,
                "tail_quiet_s": span["tail_blocks"] * blk_s, "trim_start_frame": span["a"], "trim_end_frame": span["b"],
                "trimmed_s": (span["b"] - span["a"]) * FRAME / sr})
    EXT_DATA[fid] = {"P": P, "ms": ms, "span": span}
    _rows.append(row)
    if (i + 1) % 50 == 0 or i + 1 == len(EXT_WAVS):
        print(f"  {i + 1}/{len(EXT_WAVS)} files | {time.time() - _t0:.0f} s")
EXT_META = pd.DataFrame(_rows).set_index("file_id")
EXT_META.to_csv(OUT / "external_metadata.csv")
print(f"Saved {OUT / 'external_metadata.csv'} ({len(EXT_META)} rows)")
print(f"Folder levels discovered: extra levels {sorted(set(EXT_META.extra_levels))} | file names "
      f"{sorted(set(EXT_META.file_name))} | configurations per class "
      f"{EXT_META.groupby('class').configuration.nunique().to_dict()}")

In [ ]:
# ==============================================================================
# 1b. EXTERNAL HOOKS: counts/formats (HARD), facts 1-5 (soft), per-configuration steady-RMS table
# ==============================================================================
E = EXPECTED_EXT
_cls_counts = np.bincount(EXT_META["class"], minlength=N_CLASSES).tolist()
hook("external: file count + class counts", len(EXT_META) == E["n_files"] and _cls_counts == E["class_counts"],
     f"{len(EXT_META)} files, classes {_cls_counts} (expected {E['n_files']}, {E['class_counts']})", hard=True)
_fmt = EXT_META[["n_channels", "sample_rate", "sample_width_bytes"]].value_counts().to_dict()
hook("external: formats mono / 16 kHz / 16-bit, every file decoded",
     bool((EXT_META.n_channels == 1).all() and (EXT_META.sample_rate == E["rate"]).all()
          and (EXT_META.sample_width_bytes == E["sample_width_bytes"]).all() and (EXT_META.status == "ok").all()),
     f"(channels, rate, bytes) -> files: {_fmt}; status {EXT_META.status.value_counts().to_dict()}", hard=True)
_cfg_bits_ok = all(r["configuration"].count("1") == r["class"] for _, r in EXT_META.iterrows())
hook("external: configuration bits match the class (number of 1s = broken propellers)", _cfg_bits_ok,
     f"configurations {sorted(EXT_META.configuration.unique())}")

_d = EXT_META.duration_s
_tot_h = _d.sum() / 3600
hook("external: total duration", abs(_tot_h - E["total_h"]) <= E["total_h_tol"],
     f"{_tot_h:.3f} h (expected {E['total_h']} ± {E['total_h_tol']})")
hook("external: duration range and median",
     abs(_d.min() - E["dur_min_s"]) <= 0.2 and abs(_d.max() - E["dur_max_s"]) <= 0.2
     and abs(_d.median() - E["dur_median_s"]) <= 0.5,
     f"{_d.min():.1f}–{_d.max():.1f} s, median {_d.median():.1f} s (expected {E['dur_min_s']}–{E['dur_max_s']}, "
     f"median {E['dur_median_s']})")
_dup = EXT_META.sha1.duplicated(keep=False)
hook("external: no duplicate files (sha1 of the audio bytes)", not _dup.any(),
     f"{int(_dup.sum())} files share a hash" + (f": {EXT_META.path[_dup].tolist()}" if _dup.any() else ""))
hook("external: no clipping, no dead files, |DC| <= 0.6 LSB, <= 0.8% exact-zero samples",
     EXT_META.n_clipped.sum() == 0 and (EXT_META["std"] > 0).all() and EXT_META.dc_mean.abs().max() <= E["max_abs_dc"]
     and EXT_META.zero_frac.max() <= E["max_zero_frac"],
     f"clipped samples {int(EXT_META.n_clipped.sum())}, dead {int((EXT_META['std'] == 0).sum())}, max |DC| "
     f"{EXT_META.dc_mean.abs().max():.2f} LSB, max zero fraction {100 * EXT_META.zero_frac.max():.2f}%")
_lead, _tail = EXT_META.lead_quiet_s, EXT_META.tail_quiet_s
hook("external: already trimmed (quiet lead-in median ~1 s / max ~6 s; tail median ~2 s / max ~9 s)",
     abs(_lead.median() - E["lead_median_s"]) <= 1.0 and abs(_tail.median() - E["tail_median_s"]) <= 1.0
     and abs(_lead.max() - E["lead_max_s"]) <= 2.0 and abs(_tail.max() - E["tail_max_s"]) <= 2.0,
     f"lead-in median {_lead.median():.1f} s (max {_lead.max():.1f}), tail median {_tail.median():.1f} s "
     f"(max {_tail.max():.1f}); quiet = 1-s blocks below 0.5 x the file's steady RMS")

_cls_med = EXT_META.groupby("class").steady_rms.median().reindex(range(N_CLASSES))
_rel = (_cls_med.values - np.array(E["steady_rms_class"])) / np.array(E["steady_rms_class"])
hook("external: class median steady RMS within 10% of the earlier audit", bool((np.abs(_rel) <= E["rms_rel_tol"]).all()),
     " | ".join(f"{CLASS_NAMES[c]} {_cls_med[c]:.0f} (exp {E['steady_rms_class'][c]}, {100 * _rel[c]:+.0f}%)"
                for c in range(N_CLASSES)))

print("\nPer-configuration steady-flight RMS (int16 units; steady = median 1-s block RMS over the middle 50% of the file):")
_cfg = EXT_META.groupby(["class", "configuration"]).steady_rms.agg(
    n="size", median="median", q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75), min="min", max="max")
_cfg["IQR/median"] = (_cfg.q75 - _cfg.q25) / _cfg["median"]
_cfg["expected"] = [E["steady_rms_config"].get(cfg, np.nan) for _, cfg in _cfg.index]
_cfg["dev vs expected"] = (_cfg["median"] - _cfg.expected) / _cfg.expected
print(_cfg.round(3).to_string())
_cfg_dev = _cfg["dev vs expected"].dropna()
hook("external: per-configuration median steady RMS within 10% of the earlier audit",
     bool((_cfg_dev.abs() <= E["rms_rel_tol"]).all()),
     f"max deviation {100 * _cfg_dev.abs().max():.0f}% over {len(_cfg_dev)} configurations with a recorded value")


def _run_num(s):
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else np.nan


_rho = []
for (cfg, mt), g in EXT_META.groupby(["configuration", "mission_type"]):
    if len(g) >= 4:
        _rho.append(pd.Series(g.run.map(_run_num).values).corr(pd.Series(g.steady_rms.values), method="spearman"))
print(f"\nDrift across runs (Spearman of run number vs steady RMS within configuration x mission type, groups with >= 4 "
      f"files): median rho {np.nanmedian(_rho):+.2f}, |rho| > 0.7 in {int(np.nansum(np.abs(_rho) > 0.7))}/{len(_rho)} groups "
      f"(descriptive only)")
_r1, _r2 = EXT_META.steady_rms[EXT_META["class"] == 1], EXT_META.steady_rms[EXT_META["class"] == 2]
_r3, _r4 = EXT_META.steady_rms[EXT_META["class"] == 3].values, EXT_META.steady_rms[EXT_META["class"] == 4].values
_f2b = float((_r2 < _r1.median()).mean())
_p43 = float((_r4[:, None] > _r3[None, :]).mean())
hook("external: loudness overlap facts",
     abs(_f2b - E["frac_2b_quieter_than_median_1b"]) <= 0.05 and abs(_p43 - E["p_4b_louder_than_3b"]) <= 0.05,
     f"{100 * _f2b:.0f}% of 2_broken files are quieter than the median 1_broken file (expected 17%); a random 4_broken "
     f"file is louder than a random 3_broken file {100 * _p43:.0f}% of the time (expected 75%)")

fig, ax = plt.subplots(figsize=(11, 3.8))
_order = list(_cfg.index)
for j, (c, cfg) in enumerate(_order):
    v = EXT_META.steady_rms[(EXT_META["class"] == c) & (EXT_META.configuration == cfg)].values
    ax.scatter(np.full(len(v), j) + np.linspace(-0.15, 0.15, len(v)), v, s=12, color=CLASS_COLORS[c],
               label=CLASS_NAMES[c] if cfg == _cfg.loc[c].index[0] else None)
ax.set_xticks(range(len(_order)))
ax.set_xticklabels([f"{cfg}" for _, cfg in _order], rotation=60)
ax.set_yscale("log")
ax.set_ylabel("steady-flight RMS (int16 units, log)")
ax.set_xlabel("configuration (1 = broken propeller at that position)")
ax.legend(ncol=5)
ax.set_title("External dataset: steady-flight loudness by propeller configuration")
save_fig(fig, "1b_external_steady_rms_by_configuration",
         "One dot per recording. Steady RMS = median 1-s block RMS over the middle 50% of the file (16 kHz, known). "
         "Loudness depends on WHICH propeller is broken, not only on how many.")

## 2. TII audio
- Only `AudioBuffer` files are read. Each line's `data` holds 1536 ints = 256 frames × 6 interleaved slots. Only lines
  of the modal length 1536 are kept. Slot index 5 (identically zero) is dropped, which leaves 5 channels (c1–c5 = slots
  0–4).
- The 4 missions with a corrupt `recording_started_time` are excluded by name. Days come from the mission name, with
  2021-12-29 merged into 2021-12-31.
- **Parsed once and cached per mission** (`audio_zeroshot_cache/tii/<mission>.npz`):
  - per-chunk, per-channel rFFT power, bins 0–64, after removing each channel's per-chunk mean (rectangular window);
  - per-chunk, per-channel RMS.
- **Trimming** uses the same rule as section 1. One trim span per mission is computed from the channel-mean power, so
  the 5 channels stay aligned. ~1 s of chunks = 62 chunks under the **assumed** 16 kHz rate.
- **Steady RMS per channel** = the median ~1-s block RMS over the middle 50% of the mission.
- Saved: `tii_audio_missions.csv`.

In [ ]:
# ==============================================================================
# 2a. TII: LIST MISSIONS, PARSE EACH AudioBuffer ONCE -> cache
# ==============================================================================
def mission_day(name):
    d = name[8:18].replace("_", "-")               # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


_all = []
for cdir in sorted(TII_ROOT.iterdir()):
    if not (cdir.is_dir() and cdir.name.isdigit()):
        continue
    for mdir in sorted(cdir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            _all.append((mdir.name, int(cdir.name), sorted(mdir.glob("*AudioBuffer*.jsonl"))))
TII_EXCLUDED_FOUND = sorted(n for n, _, _ in _all if n in TII_EXCLUDED)
_missing = [n for n, _, f in _all if n not in TII_EXCLUDED and not f]
TII_LIST = sorted((n, c, f[0]) for n, c, f in _all if n not in TII_EXCLUDED and f)
print(f"Mission folders with an AudioBuffer file: {len(_all)} | excluded by name: {len(TII_EXCLUDED_FOUND)} "
      f"{TII_EXCLUDED_FOUND} | kept: {len(TII_LIST)} | kept folders without an AudioBuffer file: {_missing}")


def parse_tii(name, path):
    cache = TII_CACHE / f"{name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}, "cached"
    chunks, lens, n_bad = [], Counter(), 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                n_bad += 1
                continue
            x = rec.get("data")
            if x is None:
                n_bad += 1
                continue
            lens[len(x)] += 1
            if len(x) == BUFFER_LEN:
                chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{name}: modal buffer length {modal} != {BUFFER_LEN}"
    raw = np.asarray(chunks, dtype=np.int64)        # (n, 1536)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)   # sample i -> frame i // 6, slot i % 6
    slot_vals = frames.reshape(-1, N_SLOTS)
    P = np.empty((len(raw), N_CH, N_CACHE_BINS), np.float32)
    R = np.empty((len(raw), N_CH), np.float32)
    for j, s in enumerate(KEEP_SLOTS):
        P[:, j, :], R[:, j] = frame_spectra(frames[:, :, s].astype(np.float64))
    out = {"P": P, "rms": R, "slot_zero": np.all(slot_vals == 0, axis=0), "slot_std": slot_vals.std(axis=0),
           "slot_min": slot_vals.min(axis=0), "slot_max": slot_vals.max(axis=0), "n_chunks": len(raw),
           "n_dropped": sum(v for k, v in lens.items() if k != BUFFER_LEN), "n_bad_lines": n_bad, "modal_len": modal}
    tmp = TII_CACHE / f"{name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)                          # atomic: an interrupted parse never leaves a bad cache
    return out, "parsed"


TII, _status = {}, Counter()
_t0 = time.time()
B_TII = block_frames(ASSUMED_TII_RATE)              # ~1 s of chunks UNDER THE ASSUMED RATE
for i, (name, cls, path) in enumerate(TII_LIST, 1):
    z, st = parse_tii(name, path)
    _status[st] += 1
    ms = z["rms"].astype(np.float64) ** 2           # (n, 5)
    span = trim_span(ms.mean(axis=1), B_TII)        # one span per mission (channel-mean), channels stay aligned
    assert span is not None, f"{name}: silent or too short to trim"
    steady = []
    for c in range(N_CH):
        br = block_rms(ms[:, c], B_TII)
        steady.append(float(np.median(br[len(br) // 4: max(len(br) // 4 + 1, (3 * len(br)) // 4)])))
    TII[name] = {"class": cls, "day": mission_day(name), "P": z["P"], "ms": ms, "span": span,
                 "steady_rms": np.array(steady), **{k: z[k] for k in ("slot_zero", "slot_std", "slot_min", "slot_max")},
                 "n_chunks": int(z["n_chunks"]), "n_dropped": int(z["n_dropped"]), "n_bad_lines": int(z["n_bad_lines"])}
    if i % 10 == 0 or i == len(TII_LIST):
        print(f"  {i}/{len(TII_LIST)} missions | {dict(_status)} | {time.time() - _t0:.0f} s")
MISSIONS = sorted(TII)

In [ ]:
# ==============================================================================
# 2b. TII HOOKS (HARD: 95 missions, class/day counts, slot 5 zero) + tii_audio_missions.csv + ranges
# ==============================================================================
_cls = np.bincount([TII[m]["class"] for m in MISSIONS], minlength=N_CLASSES).tolist()
_days = [sum(TII[m]["day"] == d for m in MISSIONS) for d in DAYS]
hook("TII: 95 missions, class counts, exclusion set",
     len(MISSIONS) == EXPECTED_TII_N and _cls == EXPECTED_TII_CLASS_COUNTS and set(TII_EXCLUDED_FOUND) == TII_EXCLUDED,
     f"{len(MISSIONS)} missions, classes {_cls}, excluded found {len(TII_EXCLUDED_FOUND)}/4 "
     f"(expected {EXPECTED_TII_N}, {EXPECTED_TII_CLASS_COUNTS})", hard=True)
hook("TII: day counts", _days == EXPECTED_DAY_COUNTS, f"{dict(zip(DAYS, _days))} (expected {EXPECTED_DAY_COUNTS})",
     hard=True)
_z5 = sum(bool(TII[m]["slot_zero"][DROP_SLOT]) for m in MISSIONS)
hook("TII: slot index 5 identically zero", _z5 == len(MISSIONS), f"{_z5}/{len(MISSIONS)} missions", hard=True)
_loudest0 = sum(int(np.argmax(TII[m]["slot_std"])) == 0 for m in MISSIONS)
hook("TII: slot 0 has the largest std (earlier: 94/95)", _loudest0 >= 90, f"{_loudest0}/{len(MISSIONS)} missions")

_tii_rows = []
for m in MISSIONS:
    d = TII[m]
    sp = d["span"]
    blk_s = B_TII * FRAME / ASSUMED_TII_RATE
    _tii_rows.append({"mission": m, "class": d["class"], "day": d["day"], "n_chunks": d["n_chunks"],
                      "n_dropped_nonmodal": d["n_dropped"], "n_bad_lines": d["n_bad_lines"],
                      "trim_start_chunk": sp["a"], "trim_end_chunk": sp["b"],
                      "lead_quiet_s_assumed16k": sp["lead_blocks"] * blk_s, "tail_quiet_s_assumed16k": sp["tail_blocks"] * blk_s,
                      **{f"steady_rms_c{c + 1}": d["steady_rms"][c] for c in range(N_CH)},
                      **{f"slot{s}_min": int(d["slot_min"][s]) for s in range(N_SLOTS)},
                      **{f"slot{s}_max": int(d["slot_max"][s]) for s in range(N_SLOTS)}})
TII_META = pd.DataFrame(_tii_rows).set_index("mission")
TII_META.to_csv(OUT / "tii_audio_missions.csv")
print(f"Saved {OUT / 'tii_audio_missions.csv'}")
print(f"Non-modal chunks dropped: {int(TII_META.n_dropped_nonmodal.sum())} | undecodable lines: "
      f"{int(TII_META.n_bad_lines.sum())} | chunks per mission: median {TII_META.n_chunks.median():.0f}, "
      f"range {TII_META.n_chunks.min()}–{TII_META.n_chunks.max()}")
print(f"Trim (assumed 16 kHz): quiet lead-in median {TII_META.lead_quiet_s_assumed16k.median():.1f} s "
      f"(max {TII_META.lead_quiet_s_assumed16k.max():.1f}), tail median {TII_META.tail_quiet_s_assumed16k.median():.1f} s "
      f"(max {TII_META.tail_quiet_s_assumed16k.max():.1f})")

print("\nTII sample range per slot over all missions (raw integers; int16 is NOT assumed):")
for s in range(N_SLOTS):
    lab = f"c{KEEP_SLOTS.index(s) + 1}" if s in KEEP_SLOTS else "dropped"
    print(f"  slot {s} ({lab}): min {TII_META[f'slot{s}_min'].min()}, max {TII_META[f'slot{s}_max'].max()}")
_norm = TII_META[TII_META["class"] == 0]
print("Median steady RMS of Normal (class 0) missions per channel (TII units): " +
      ", ".join(f"c{c + 1} {_norm[f'steady_rms_c{c + 1}'].median():.1f}" for c in range(N_CH)))
print("Day x class (missions):")
print(pd.crosstab(TII_META.day, TII_META["class"]).reindex(index=DAYS).to_string())

## 3. Compatibility audit (descriptive; printed before any model trains; nothing here stops the notebook)
- **3a — relative loudness pattern.** For each class, the median steady RMS ÷ the class-0 median, for the external
  data and for each TII channel. Is the TII pattern also monotone and saturating?
- **3b — sample-rate scan.** This is **exploratory, not a verification**: a match also requires a similar drone
  state (rotor speed, load).
  - For each candidate TII per-channel rate, TII bin *k* is mapped to *k* × rate / 256 Hz.
  - The mean log-power of steady Normal frames is interpolated onto a 100–2000 Hz grid (25 Hz steps) and correlated
    with the external Normal mean log-power (16 kHz, known).
  - `RATE_EST` = the best rate, but only if it beats the second best by ≥ 0.05 (mean r over its 3 best channels).
    Otherwise the result is "inconclusive" and 16 kHz is kept.
- **3c — channel ranking** by similarity to the external `ch1` in 3a and 3b. Descriptive only.
- **3d — overlap screen.** For each TII mission, the highest correlation of its time-normalized 1-s RMS envelope with
  any external file of the same class, against the same statistic for other TII missions (the baseline). Pairs
  > 0.98 are flagged. This is a **soft** screen: shared flight shapes alone produce high correlations.

In [ ]:
# ==============================================================================
# 3a. RELATIVE LOUDNESS PATTERN (class median steady RMS / class-0 median)
# ==============================================================================
_ext_med = EXT_META.groupby("class").steady_rms.median().reindex(range(N_CLASSES))
LOUD = pd.DataFrame({"external ch1": _ext_med / _ext_med[0]})
for c in range(N_CH):
    _m = TII_META.groupby("class")[f"steady_rms_c{c + 1}"].median().reindex(range(N_CLASSES))
    LOUD[f"TII c{c + 1}"] = _m / _m[0]
LOUD.index = CLASS_NAMES
print("Steady-flight median RMS per class divided by the class-0 median:")
print(LOUD.round(2).to_string())


def _pattern(r):
    lr = np.log(np.asarray(r, float))
    monotone = bool(np.all(np.diff(lr) > 0))
    saturating = bool((lr[4] - lr[2]) < (lr[2] - lr[0]))      # the rise from 2 to 4 broken < the rise from 0 to 2
    return monotone, saturating


LOUD_PATTERN = {col: _pattern(LOUD[col].values) for col in LOUD.columns}
for col, (mono, sat) in LOUD_PATTERN.items():
    print(f"  {col:<13} monotone increasing: {mono!s:<5} | saturating (rise 2->4 < rise 0->2, log scale): {sat}")
_ext_pat = LOUD_PATTERN["external ch1"]
_similar = [col for col in LOUD.columns[1:] if LOUD_PATTERN[col] == _ext_pat]
print(f"\nSTATEMENT: the external data is {'monotone' if _ext_pat[0] else 'NOT monotone'} and "
      f"{'saturating' if _ext_pat[1] else 'NOT saturating'}. TII channels with the same qualitative pattern: "
      f"{_similar if _similar else 'none'}.")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(range(N_CLASSES), LOUD["external ch1"], color=EXT_COLOR, lw=2.5, marker="o", label="external ch1 (16 kHz)")
for c in range(N_CH):
    ax.plot(range(N_CLASSES), LOUD[f"TII c{c + 1}"], color=CH_COLORS[c], marker="s", ls="--", label=f"TII c{c + 1}")
ax.set_yscale("log")
ax.set_xticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("median steady RMS / class-0 median (log)")
ax.legend(ncol=2)
ax.set_title("Relative loudness by number of broken propellers")
save_fig(fig, "3a_relative_loudness",
         "Steady RMS = median ~1-s block RMS over the middle 50% of each recording; TII blocks are 62 chunks (~1 s only "
         "under the assumed 16 kHz rate). Ratios are unit-free, so different gains do not matter here.")

In [ ]:
# ==============================================================================
# 3b. SAMPLE-RATE SCAN (exploratory): TII Normal spectrum vs external Normal spectrum on a common Hz grid
# ==============================================================================
def mean_steady_logpow(P):
    """P: (n, ..., 65). Mean over the middle 50% of frames of log(power)."""
    n = len(P)
    return np.log(P[n // 4: max(n // 4 + 1, (3 * n) // 4)].astype(np.float64) + LOGPOW_EPS).mean(axis=0)


EXT_NORMAL_LOGPOW = np.mean([mean_steady_logpow(EXT_DATA[f]["P"]) for f in EXT_META.index[EXT_META["class"] == 0]], axis=0)
TII_NORMAL_LOGPOW = np.mean([mean_steady_logpow(TII[m]["P"]) for m in MISSIONS if TII[m]["class"] == 0], axis=0)  # (5, 65)


def to_grid(logpow, hz_per_bin):
    """Interpolate bins 1..64 (DC excluded) onto SCAN_GRID_HZ; grid points outside the covered range are NaN."""
    f = np.arange(1, N_CACHE_BINS) * hz_per_bin
    g = np.full(len(SCAN_GRID_HZ), np.nan)
    ok = (SCAN_GRID_HZ >= f[0]) & (SCAN_GRID_HZ <= f[-1])
    g[ok] = np.interp(SCAN_GRID_HZ[ok], f, logpow[1:])
    return g


EXT_GRID = to_grid(EXT_NORMAL_LOGPOW, EXT_RATE / FRAME)
_rows = []
for r in RATE_CANDIDATES:
    for c in range(N_CH):
        g = to_grid(TII_NORMAL_LOGPOW[c], r / FRAME)
        ok = np.isfinite(g) & np.isfinite(EXT_GRID)
        corr = float(np.corrcoef(g[ok], EXT_GRID[ok])[0, 1]) if ok.sum() >= 10 else np.nan
        _rows.append({"rate_hz": r, "channel": f"c{c + 1}", "pearson_r": corr, "n_grid_points": int(ok.sum())})
RATE_SCAN = pd.DataFrame(_rows)
RATE_SCAN.to_csv(OUT / "compat_rate_scan.csv", index=False)
_piv = RATE_SCAN.pivot(index="rate_hz", columns="channel", values="pearson_r")
_top = _piv.apply(lambda row: np.sort(row.dropna().values)[::-1][:RATE_TOP_CHANNELS].mean(), axis=1)
print("Pearson r between TII Normal mean log-power and external Normal mean log-power, 100-2000 Hz grid:")
print(_piv.assign(**{f"mean of best {RATE_TOP_CHANNELS} channels": _top}).round(3).to_string())
_rank = _top.sort_values(ascending=False)
_best, _second = _rank.index[0], _rank.index[1]
if _rank.iloc[0] - _rank.iloc[1] >= RATE_MARGIN:
    RATE_EST = int(_best)
    print(f"\nRATE_EST = {RATE_EST} Hz (top-{RATE_TOP_CHANNELS} mean r {_rank.iloc[0]:.3f} vs second best {_second} Hz "
          f"{_rank.iloc[1]:.3f}; margin >= {RATE_MARGIN})")
else:
    RATE_EST = "inconclusive"
    print(f"\nRATE_EST = inconclusive (best {_best} Hz r {_rank.iloc[0]:.3f}, second {_second} Hz r {_rank.iloc[1]:.3f}; "
          f"margin < {RATE_MARGIN}). The assumed {ASSUMED_TII_RATE} Hz is kept.")
print("This scan is EXPLORATORY. It assumes the two datasets' Normal flights have a similar drone state (rotor speed, "
      "load). It is NOT a verification of the TII sample rate.")
RATE_VARIANTS = [ASSUMED_TII_RATE]
if isinstance(RATE_EST, int) and abs(RATE_EST - ASSUMED_TII_RATE) / ASSUMED_TII_RATE > RATE_VARIANT_MIN_REL:
    RATE_VARIANTS.append(RATE_EST)
RATE_FOR_RANK = RATE_EST if isinstance(RATE_EST, int) else ASSUMED_TII_RATE
print(f"Rate variants used for TII inputs: {RATE_VARIANTS}")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for c in range(N_CH):
    axs[0].plot(_piv.index, _piv[f"c{c + 1}"], marker="o", color=CH_COLORS[c], label=f"TII c{c + 1}")
axs[0].plot(_top.index, _top.values, color=EXT_COLOR, lw=2.5, label=f"mean of best {RATE_TOP_CHANNELS}")
axs[0].axvline(ASSUMED_TII_RATE, color="#52514e", ls="--", lw=1)
axs[0].set_xlabel("candidate TII per-channel rate (Hz)")
axs[0].set_ylabel("Pearson r with external Normal spectrum")
axs[0].legend(ncol=2)
axs[0].set_title("Rate scan (exploratory)")
_zs = lambda v: (v - np.nanmean(v)) / np.nanstd(v)
axs[1].plot(SCAN_GRID_HZ, _zs(EXT_GRID), color=EXT_COLOR, lw=2.5, label="external ch1 (16 kHz)")
for c in range(N_CH):
    axs[1].plot(SCAN_GRID_HZ, _zs(to_grid(TII_NORMAL_LOGPOW[c], RATE_FOR_RANK / FRAME)), color=CH_COLORS[c], lw=1,
                label=f"TII c{c + 1}")
axs[1].set_xlabel(f"Hz (TII mapped with {RATE_FOR_RANK} Hz)")
axs[1].set_ylabel("mean log-power, z-scored")
axs[1].legend(ncol=2)
axs[1].set_title("Normal-class steady spectra on the common grid")
save_fig(fig, "3b_rate_scan",
         "Exploratory: correlation depends on the two datasets having a similar drone state. " + RATE_CAVEAT +
         " DC bin excluded; steady = middle 50% of each recording's frames.")

In [ ]:
# ==============================================================================
# 3c. CHANNEL RANKING vs external ch1 (descriptive only)
# ==============================================================================
_lr_ext = np.log(LOUD["external ch1"].values[1:])
_rank_rows = []
for c in range(N_CH):
    _rank_rows.append({"channel": f"c{c + 1}",
                       "loudness pattern distance (mean |log ratio diff|, classes 1-4)":
                           float(np.mean(np.abs(np.log(LOUD[f"TII c{c + 1}"].values[1:]) - _lr_ext))),
                       f"spectral r at {RATE_FOR_RANK} Hz": float(_piv.loc[RATE_FOR_RANK, f"c{c + 1}"])})
CH_RANK = pd.DataFrame(_rank_rows).set_index("channel")
CH_RANK["rank: loudness"] = CH_RANK.iloc[:, 0].rank().astype(int)
CH_RANK["rank: spectrum"] = CH_RANK.iloc[:, 1].rank(ascending=False).astype(int)
CH_RANK["mean rank"] = CH_RANK[["rank: loudness", "rank: spectrum"]].mean(axis=1)
CH_RANK = CH_RANK.sort_values("mean rank")
print("TII channels ranked by similarity to external ch1 (1 = most similar). DESCRIPTIVE ONLY: no arm is chosen from this.")
print(CH_RANK.round(3).to_string())

In [ ]:
# ==============================================================================
# 3d. OVERLAP SCREEN (soft): time-normalized 1-s RMS envelopes, TII vs external of the same class
# ==============================================================================
def envelope(span):
    seg = span["block_rms"][span["lead_blocks"]: span["n_blocks"] - span["tail_blocks"]]
    if len(seg) < 3:
        return None
    e = np.interp(np.linspace(0, 1, ENVELOPE_POINTS), np.linspace(0, 1, len(seg)), seg)
    return (e - e.mean()) / (e.std() + 1e-12)


_ext_ids = [f for f in EXT_META.index if envelope(EXT_DATA[f]["span"]) is not None]
Z_EXT = np.stack([envelope(EXT_DATA[f]["span"]) for f in _ext_ids])
Z_TII = np.stack([envelope(TII[m]["span"]) for m in MISSIONS])
C_TE = Z_TII @ Z_EXT.T / ENVELOPE_POINTS             # Pearson r (vectors are z-scored with the population std)
C_TT = Z_TII @ Z_TII.T / ENVELOPE_POINTS
_ext_cls = EXT_META.loc[_ext_ids, "class"].values
_tii_cls = np.array([TII[m]["class"] for m in MISSIONS])
_ov, _flags = [], []
for i, m in enumerate(MISSIONS):
    same_e = np.where(_ext_cls == _tii_cls[i])[0]
    same_t = np.array([j for j in np.where(_tii_cls == _tii_cls[i])[0] if j != i])
    je = same_e[np.argmax(C_TE[i, same_e])]
    _ov.append({"mission": m, "class": _tii_cls[i], "max r vs external (same class)": C_TE[i, je],
                "best external file": EXT_META.loc[_ext_ids[je], "path"],
                "max r vs other TII (same class, baseline)": C_TT[i, same_t].max()})
    _flags += [(m, EXT_META.loc[_ext_ids[j], "path"], C_TE[i, j]) for j in same_e if C_TE[i, j] > OVERLAP_FLAG]
OVERLAP = pd.DataFrame(_ov).set_index("mission")
OVERLAP.to_csv(OUT / "compat_overlap_screen.csv")
print(OVERLAP.groupby("class")[["max r vs external (same class)", "max r vs other TII (same class, baseline)"]]
      .median().round(3).to_string())
print(f"\nMedian over missions: max r vs external {OVERLAP.iloc[:, 1].median():.3f} | baseline (other TII) "
      f"{OVERLAP.iloc[:, 3].median():.3f}")
print(f"TII-external pairs with r > {OVERLAP_FLAG}: {len(_flags)}")
for m, p, r in sorted(_flags, key=lambda t: -t[2])[:20]:
    print(f"  FLAG {m} <-> {p}: r {r:.3f}")
print("SOFT SCREEN ONLY: shared flight shapes (take-off, waypoints, landing) alone produce high correlations, so a flag "
      "is a reason to look, not evidence of a shared recording.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
_bins = np.linspace(min(OVERLAP.iloc[:, 1].min(), OVERLAP.iloc[:, 3].min()), 1, 30)
ax.hist(OVERLAP.iloc[:, 3], bins=_bins, alpha=0.6, color=CH_COLORS[1], label="vs other TII missions (baseline)")
ax.hist(OVERLAP.iloc[:, 1], bins=_bins, alpha=0.6, color=CH_COLORS[0], label="vs external files")
ax.axvline(OVERLAP_FLAG, color="#52514e", ls="--", lw=1)
ax.set_xlabel("max Pearson r of the 1-s RMS envelope (same class)")
ax.set_ylabel("TII missions")
ax.legend()
ax.set_title("Overlap screen")
save_fig(fig, "3d_overlap_screen",
         f"Envelopes cover each trimmed span, resampled to {ENVELOPE_POINTS} points and z-scored. TII blocks are ~1 s only "
         f"under the assumed 16 kHz rate. Soft screen: similar flight shapes alone give high r.")

In [ ]:
# ==============================================================================
# 3e. COMPATIBILITY SUMMARY (printed before any model trains)
# ==============================================================================
COMPAT_LINES = [
    "COMPATIBILITY FINDINGS",
    f"- Loudness pattern: external is {'monotone' if _ext_pat[0] else 'not monotone'} and "
    f"{'saturating' if _ext_pat[1] else 'not saturating'}; TII channels with the same pattern: {_similar or 'none'}.",
    f"- Sample-rate scan (exploratory): RATE_EST = {RATE_EST}; rate variants used: {RATE_VARIANTS}.",
    f"- Channel ranking (most to least similar to external ch1): {list(CH_RANK.index)}.",
    f"- Overlap screen: median max r vs external {OVERLAP.iloc[:, 1].median():.3f} (baseline vs other TII "
    f"{OVERLAP.iloc[:, 3].median():.3f}); {len(_flags)} TII-external pairs above {OVERLAP_FLAG} (soft screen).",
]
print("\n".join(COMPAT_LINES))

## 4. Front end (identical for both datasets; ONE channel per sample)
- **Frame = 256 samples = one chunk.** Per frame: remove the mean, take the rFFT power *P*, and use bins 1–16
  (62.5–1000 Hz at 16 kHz).
- **Two feature sets:**
  - **LEVEL** = *P*^0.25 (sqrt-magnitude, as in earlier TII work; no per-frame normalization, so loudness is kept);
  - **SHAPE** = log(*P_k* + ε) − log(Σ_{j=1..16} *P_j* + ε) (scale-free: loudness removed).
- **Windows.** Tokens = the mean of 2 consecutive frames. A window is 128 frames with stride 64, giving
  (64 tokens × 16 features). Windows lie inside one recording's trimmed span, and all windows of a recording stay in
  one split.
- **TII → external Hz grid.** The cached TII power is interpolated onto *f_j* = 62.5·*j* Hz (*j* = 1..16) using the
  assumed rate. At 16 kHz this is the identity. A `RATE_EST` variant is added only if section 3b found a rate that
  differs from 16 kHz by > 5%. **Interpolating power between bins is an approximation.**
- **Gain handling for TII inputs (LEVEL only).**
  - **RAW:** no rescaling.
  - **NORMCAL:** for a mission recorded on day *d* and each channel, *g* = the median steady RMS of the Normal TII
    missions from the **other four** days. TII power is then scaled by (rms_external_normal / *g*)².
  - NORMCAL uses TII Normal-class loudness from other days. It is a **semi-supervised gain calibration**, not pure
    zero-shot.
  - SHAPE is scale-free, so it gets no gain handling.
- Class-weighted loss. z-score statistics come from each model's **training data only**.

In [ ]:
# ==============================================================================
# 4a. FRONT-END FUNCTIONS
# ==============================================================================
def hz_weight_matrix(rate):
    """(16, 65) linear-interpolation weights from cached bins 0..64 (bin k = k * rate / 256 Hz) to TARGET_HZ. Positions
    below bin 1 are clamped to bin 1 (the DC bin is ~0 after mean removal and is never used)."""
    pos = TARGET_HZ * FRAME / rate
    n_clamped = int(((pos < 1) | (pos > N_CACHE_BINS - 1)).sum())
    pos = np.clip(pos, 1, N_CACHE_BINS - 1)
    W = np.zeros((N_FEAT, N_CACHE_BINS))
    lo = np.floor(pos).astype(int)
    frac = pos - lo
    for j in range(N_FEAT):
        W[j, lo[j]] += 1 - frac[j]
        if frac[j] > 0:
            W[j, min(lo[j] + 1, N_CACHE_BINS - 1)] += frac[j]
    return W, n_clamped


for _r in sorted(set(RATE_VARIANTS) | {EXT_RATE}):
    _W, _nc = hz_weight_matrix(_r)
    print(f"rate {_r} Hz: target bins at cached-bin positions {np.round(TARGET_HZ * FRAME / _r, 2).tolist()}"
          + (f" | {_nc} position(s) clamped to the cached range" if _nc else ""))
_W16, _ = hz_weight_matrix(16000)
_ident = np.zeros((N_FEAT, N_CACHE_BINS))
_ident[np.arange(N_FEAT), np.arange(1, N_FEAT + 1)] = 1
assert np.allclose(_W16, _ident), "at 16 kHz the Hz mapping must be the identity on bins 1..16"
print("At 16 kHz the mapping is the identity on bins 1..16 (checked).")


def level_from_power(P16):
    return np.power(np.maximum(P16, 0.0), 0.25)


def shape_from_power(P16):
    return np.log(P16 + SHAPE_EPS) - np.log(P16.sum(axis=-1, keepdims=True) + SHAPE_EPS)


FEATURE_FN = {"LEVEL": level_from_power, "SHAPE": shape_from_power}
GAINS_FOR = {"LEVEL": ["RAW", "NORMCAL"], "SHAPE": ["NONE"]}


def window_starts(n_frames):
    return list(range(0, n_frames - WINDOW_FRAMES + 1, STRIDE_FRAMES))


def make_windows(Fr):
    """Fr: (n_frames, 16) per-frame features -> (n_windows, 64, 16); token = mean of 2 consecutive frames."""
    st = window_starts(len(Fr))
    if not st:
        return np.zeros((0, N_TOKENS, N_FEAT), np.float32)
    return np.stack([Fr[s:s + WINDOW_FRAMES].reshape(N_TOKENS, TOKEN_POOL, N_FEAT).mean(axis=1) for s in st]).astype(np.float32)

In [ ]:
# ==============================================================================
# 4b. EXTERNAL WINDOWS + VALIDATION GROUPS (whole held-out configurations / mission types)
# ==============================================================================
_W_EXT, _ = hz_weight_matrix(EXT_RATE)
EXT_X = {f: [] for f in FEATURE_SETS}
_wm = []
for fid in EXT_META.index:
    d = EXT_DATA[fid]
    P16 = d["P"][d["span"]["a"]: d["span"]["b"]].astype(np.float64) @ _W_EXT.T
    for f in FEATURE_SETS:
        EXT_X[f].append(make_windows(FEATURE_FN[f](P16)))
    for i, s in enumerate(window_starts(len(P16))):
        _wm.append({"window_id": f"{fid}#{i:04d}", "recording": fid, "class": int(EXT_META.loc[fid, "class"]), "start": s})
EXT_X = {f: np.concatenate(v) for f, v in EXT_X.items()}
EXT_WMETA = pd.DataFrame(_wm)
EXT_Y = EXT_WMETA["class"].values.astype(np.int64)
del _wm
print(f"External windows: {len(EXT_WMETA)} ({EXT_WMETA.groupby('class').size().to_dict()} per class) | "
      f"shape {EXT_X[FEATURE_SETS[0]].shape} | finite: {all(np.isfinite(v).all() for v in EXT_X.values())}")
print(f"Windows per recording: median {EXT_WMETA.groupby('recording').size().median():.0f}, "
      f"min {EXT_WMETA.groupby('recording').size().min()}")

EXT_GROUP_KIND = {c: ("configuration" if EXT_META[EXT_META["class"] == c].configuration.nunique() >= 3 else "mission_type")
                  for c in range(N_CLASSES)}
EXT_META["val_group"] = [r["configuration"] if EXT_GROUP_KIND[r["class"]] == "configuration" else r["mission_type"]
                         for _, r in EXT_META.iterrows()]
print("Validation group kind per class: " + ", ".join(
    f"{CLASS_NAMES[c]}: {EXT_GROUP_KIND[c]} ({EXT_META[EXT_META['class'] == c].val_group.nunique()} groups)"
    for c in range(N_CLASSES)))


def ext_val_split(seed):
    """Per class: shuffle the groups (deterministic from the seed) and hold out whole groups until >= 15% of the
    class's files. Never holds out every group of a class."""
    rng = np.random.default_rng([seed, 17])
    val, held = [], {}
    for c in range(N_CLASSES):
        sub = EXT_META[EXT_META["class"] == c]
        groups = sorted(sub.val_group.unique())
        need = math.ceil(EXT_VAL_FRACTION * len(sub))
        chosen, n = [], 0
        for gi in rng.permutation(len(groups)):
            if n >= need:
                break
            chosen.append(groups[gi])
            n += int((sub.val_group == groups[gi]).sum())
        assert len(chosen) < len(groups), f"class {c}: all groups would be held out"
        held[c] = chosen
        val += list(sub.index[sub.val_group.isin(chosen)])
    return sorted(set(EXT_META.index) - set(val)), sorted(val), held


EXT_SPLITS = {}
for s in SEEDS:
    tr, va, held = ext_val_split(s)
    EXT_SPLITS[s] = {"train": tr, "val": va, "held": held}
    _tr_meta, _va_meta = EXT_META.loc[tr], EXT_META.loc[va]
    _no_group_overlap = all(not (set(_tr_meta.val_group[_tr_meta["class"] == c]) & set(held[c])) for c in range(N_CLASSES))
    hook(f"external seed {s}: no validation group (or recording) also in training",
         _no_group_overlap and not (set(tr) & set(va)),
         "held out: " + "; ".join(f"{CLASS_NAMES[c]} {held[c]} ({int((_va_meta['class'] == c).sum())}/"
                                  f"{int((EXT_META['class'] == c).sum())} files)" for c in range(N_CLASSES)), hard=True)

In [ ]:
# ==============================================================================
# 4c. TII WINDOWS (built on demand per feature / rate / gain), NORMCAL GAINS, SCRATCH FOLDS
# ==============================================================================
_twm, TII_WROWS = [], {}
for m in MISSIONS:
    sp = TII[m]["span"]
    st = window_starts(sp["b"] - sp["a"])
    TII_WROWS[m] = np.arange(len(_twm), len(_twm) + len(st))
    for i, s in enumerate(st):
        _twm.append({"window_id": f"{m}#{i:04d}", "mission": m, "class": TII[m]["class"], "day": TII[m]["day"], "start": s})
TII_WMETA = pd.DataFrame(_twm)
del _twm
_wpm = TII_WMETA.groupby("mission").size()
hook("TII: every mission has >= 1 window inside its trimmed span", len(_wpm) == len(MISSIONS) and _wpm.min() >= 1,
     f"{len(TII_WMETA)} windows (x {N_CH} channels); per mission median {_wpm.median():.0f}, min {_wpm.min()}", hard=True)

# ---- NORMCAL: per held-out day d, g = per-channel median steady RMS of Normal TII missions from the OTHER four days ----
EXT_NORMAL_RMS = float(EXT_META.steady_rms[EXT_META["class"] == 0].median())
NORMCAL_G, NORMCAL_SOURCES = {}, {}
_steady_cols = [f"steady_rms_c{c + 1}" for c in range(N_CH)]
for d in DAYS:
    src = TII_META[(TII_META["class"] == 0) & (TII_META.day != d)]
    NORMCAL_G[d] = src[_steady_cols].median().values.astype(np.float64)
    NORMCAL_SOURCES[d] = list(src.index)
_gain_ok = all(all(TII_META.loc[m, "day"] != d and TII_META.loc[m, "class"] == 0 for m in NORMCAL_SOURCES[d]) for d in DAYS)
hook("NORMCAL gains use other-day Normal missions only", _gain_ok,
     "; ".join(f"{d}: {len(NORMCAL_SOURCES[d])} Normal missions from other days" for d in DAYS), hard=True)
print(f"External Normal steady RMS = {EXT_NORMAL_RMS:.1f} (int16 units)")
print("NORMCAL power scale (rms_ext_normal / g)^2 per held-out day and channel:")
print(pd.DataFrame({d: (EXT_NORMAL_RMS / NORMCAL_G[d]) ** 2 for d in DAYS}, index=[f"c{c + 1}" for c in range(N_CH)]).T
      .to_string(float_format=lambda v: f"{v:.4g}"))
print("NORMCAL is a SEMI-SUPERVISED gain calibration (it uses TII Normal-class loudness from training days), not pure "
      "zero-shot.")

TII_WIN_CACHE = {}


def tii_windows(feature, rate, gain):
    """(n_windows, 5, 64, 16) float32, rows aligned with TII_WMETA. gain: RAW / NORMCAL (LEVEL) or NONE (SHAPE)."""
    assert gain in GAINS_FOR[feature], f"{feature} does not take gain {gain}"
    key = (feature, rate, gain)
    if key in TII_WIN_CACHE:
        return TII_WIN_CACHE[key]
    while len(TII_WIN_CACHE) >= 3:                  # bounded memory
        TII_WIN_CACHE.pop(next(iter(TII_WIN_CACHE)))
    W, _ = hz_weight_matrix(rate)
    X = np.empty((len(TII_WMETA), N_CH, N_TOKENS, N_FEAT), np.float32)
    for m in MISSIONS:
        d, sp = TII[m], TII[m]["span"]
        for c in range(N_CH):
            P16 = d["P"][sp["a"]: sp["b"], c, :].astype(np.float64) @ W.T
            if gain == "NORMCAL":
                P16 = P16 * (EXT_NORMAL_RMS / NORMCAL_G[d["day"]][c]) ** 2
            X[TII_WROWS[m], c] = make_windows(FEATURE_FN[feature](P16))
    assert np.isfinite(X).all(), f"non-finite TII features for {key}"
    TII_WIN_CACHE[key] = X
    return X


# ---- TII-from-scratch day-out folds: v27/v28 fallback inner-validation rule (per class ~15%, min 1, seed [2026, k]) ----
def inner_val(pool, k):
    rng = np.random.default_rng([INNER_VAL_SEED, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if TII[m]["class"] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return sorted(train), sorted(val)


SCRATCH_FOLDS = {}
for k, day in enumerate(DAYS):
    test = sorted(m for m in MISSIONS if TII[m]["day"] == day)
    tr, va = inner_val([m for m in MISSIONS if TII[m]["day"] != day], k)
    SCRATCH_FOLDS[k] = {"day": day, "train": tr, "val": va, "test": test}
_leak = all(not (set(f["train"]) | set(f["val"])) & set(f["test"]) and not (set(f["train"]) & set(f["val"]))
            and {TII[m]["day"] for m in f["train"] + f["val"]}.isdisjoint({f["day"]}) for f in SCRATCH_FOLDS.values())
_part = sorted(m for f in SCRATCH_FOLDS.values() for m in f["test"]) == MISSIONS
hook("TII scratch: no test-day mission in any fold's training/validation; test days partition the 95 missions",
     _leak and _part, " | ".join(f"fold {k} ({f['day']}): train {len(f['train'])} / val {len(f['val'])} / test "
                                 f"{len(f['test'])}" for k, f in SCRATCH_FOLDS.items()), hard=True)

## 5. Models and training
- **Architecture:** real `mamba_ssm` only, with v28's audio path (`UnimodalMamba`) and a 16-feature input:
  `Linear(16, 256) → LayerNorm → GELU → Linear(256, 256)` → modality embedding + LayerNorm → **2 pre-norm residual
  Mamba blocks** (d_state 16, d_conv 2, expand 2) → attention pooling → `Dropout(0.15) → Linear(256, 128) → GELU →
  Linear(128, 5)`.
- **Training (v28 A1):** AdamW (lr 1e-4, weight decay 1e-4), cosine annealing to 1e-6 over 20 epochs, grad clip 1.0,
  batch 32, class-weighted cross-entropy. **Checkpoint:** the epoch with the lowest validation loss (see the deviation
  note at the top).
- **EXT_LEVEL / EXT_SHAPE** — trained on external windows only. Validation = whole held-out configurations (or mission
  types, for normal and 4_broken). Validation macro-F1 is reported as *"within-external, held-out configurations"*.
- **TII_LEVEL / TII_SHAPE** (in-domain reference) — for each day-out fold, trained on the other 4 days. Each channel's
  windows are separate samples. ~15% mission-level validation hold-out. Tested on the held-out day (every channel).
- **RMS-CENTROID** (no training) — computed in section 6.
- **Resumable:** a run already in `audio_zeroshot_runs.csv` with the same config tag (and its files) is skipped.

In [ ]:
# ==============================================================================
# 5a. MODEL (v28 UnimodalMamba audio path, 16 inputs) + real-mamba check + v28 settings
# ==============================================================================
class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
                                           nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, sum(p.numel() for p in model.parameters()), n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
with torch.no_grad():
    _shape_ok = _probe(torch.zeros(2, N_TOKENS, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook("real mamba_ssm model", _ok and _shape_ok,
     f"every block is {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); "
     f"Mamba params {_n_mamba:,} (expected {EXPECTED_MAMBA_PARAMS:,}); total {N_PARAMS:,}", hard=True)
del _probe
torch.cuda.empty_cache()
_v28 = {k: globals()[k] for k in ["D_MODEL", "DROPOUT", "MAMBA_NUM_BLOCKS", "MAMBA_D_STATE", "MAMBA_D_CONV",
                                  "MAMBA_EXPAND", "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "WEIGHT_DECAY", "LR_MIN",
                                  "GRAD_CLIP_NORM"]}
hook("v28 A1 hyperparameters printed", True, ", ".join(f"{k}={v}" for k, v in _v28.items()) +
     f" | optimizer AdamW, CosineAnnealingLR stepped per epoch | v28 checkpoint rule: {V28_CHECKPOINT_RULE} | "
     f"used here: {CHECKPOINT_RULE}")

In [ ]:
# ==============================================================================
# 5b. TRAINING FUNCTION (train-only z-score, class-weighted CE, lowest-validation-loss checkpoint)
# ==============================================================================
def make_loader(X, y, shuffle, seed):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


def zscore_stats(X):
    flat = X.reshape(-1, N_FEAT)
    mu = flat.mean(axis=0, dtype=np.float64)
    sd = flat.std(axis=0, dtype=np.float64) + 1e-6
    return mu.astype(np.float32), sd.astype(np.float32)


def apply_z(X, mu, sd):
    return ((X - mu) / sd).astype(np.float32)


@torch.no_grad()
def predict_probs(model, X, mu, sd, bs=512):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(apply_z(X[i:i + bs], mu, sd)).to(device)
        out.append(F.softmax(model(xb), dim=-1).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, N_CLASSES), np.float32)


def train_model(tag, seed, Xtr, ytr, Xva, yva, verbose=True):
    t0 = time.time()
    mu, sd = zscore_stats(Xtr)                      # TRAINING data only
    Xtr_z, Xva_z = apply_z(Xtr, mu, sd), apply_z(Xva, mu, sd)
    counts = np.bincount(ytr, minlength=N_CLASSES)
    cw = (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)   # inverse class frequency (v28)
    set_all_seeds(seed)
    model = AudioMamba().to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    ok, n_params, _ = verify_model(model)
    assert ok, "not the expected real-mamba_ssm model"
    w = torch.tensor(cw, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    criterion_sum = nn.CrossEntropyLoss(weight=w, reduction="sum")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(Xtr_z, ytr.astype(np.int64), True, seed)
    va = make_loader(Xva_z, yva.astype(np.int64), False, seed)
    if verbose:
        print(f"{tag} init {init_sha1} | {n_params:,} params | windows train {len(ytr)} / val {len(yva)} | "
              f"z-score {sha16(mu, sd)} | class weights {np.round(cw, 3).tolist()}")
    best, history = {"loss": float("inf")}, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running, nb = 0.0, 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running += loss.item()
            nb += 1
        scheduler.step()
        assert math.isfinite(running), f"{tag} non-finite training loss"
        model.eval()
        probs, lsum, wsum = [], 0.0, 0.0
        with torch.no_grad():
            for x, y in va:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                logits = model(x)
                lsum += float(criterion_sum(logits, y))
                wsum += float(w[y].sum())
                probs.append(F.softmax(logits, dim=-1).float().cpu().numpy())
        Pva = np.concatenate(probs)
        vloss = lsum / max(wsum, 1e-12)             # = weighted-mean CE, same definition as the training loss
        vf1 = f1_score(yva, Pva.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)
        history.append([epoch, running / max(nb, 1), vloss, vf1])
        improved = vloss < best["loss"]
        if improved:
            best = {"loss": vloss, "f1": vf1, "epoch": epoch, "state": copy.deepcopy(model.state_dict()), "Pva": Pva}
        if verbose:
            print(f"{tag} epoch {epoch:02d}/{EPOCHS} | train CE {running / max(nb, 1):.4f} | val CE {vloss:.4f} | "
                  f"val macro-F1 {vf1:.4f}" + ("  * best (lowest val loss)" if improved else ""))
    model.load_state_dict(best["state"])
    info = {"best_epoch": best["epoch"], "best_val_loss": best["loss"], "best_val_macro_f1_window": best["f1"],
            "Pva": best["Pva"], "mu": mu, "sd": sd, "class_weights": cw, "zscore_hash": sha16(mu, sd),
            "init_sha1": init_sha1, "n_params": n_params, "history": history, "train_seconds": time.time() - t0,
            "state": best["state"]}
    del tr, va, Xtr_z, Xva_z, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return model, info

In [ ]:
# ==============================================================================
# 5c. RUNS: EXT_{LEVEL,SHAPE} x seeds, TII_{LEVEL,SHAPE} x 5 day-out folds x seeds (resumable, config-tagged)
# ==============================================================================
CONFIG_BASE = {"notebook": "audio-audit-zeroshot v1", "frame": FRAME, "target_hz": TARGET_HZ.tolist(),
               "window": [WINDOW_FRAMES, STRIDE_FRAMES, TOKEN_POOL], "trim": [TRIM_REL, RMS_BLOCK_S],
               "eps": [SHAPE_EPS], "model": [D_MODEL, DROPOUT, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND],
               "train": [EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, LR_MIN, GRAD_CLIP_NORM],
               "checkpoint": CHECKPOINT_RULE, "ext_val": EXT_VAL_FRACTION, "inner_val": [INNER_VAL_FRACTION, INNER_VAL_SEED],
               "tii_scratch_rate": ASSUMED_TII_RATE}
ARM_TAG = {a: config_tag({**CONFIG_BASE, "arm": a}) for f in FEATURE_SETS for a in (f"EXT_{f}", f"TII_{f}")}
RUN_FIELDS = ["arm", "fold", "fold_day", "seed", "config_tag", "data_sources", "n_train_recordings", "n_val_recordings",
              "n_test_recordings", "n_train_windows", "n_val_windows", "n_test_windows", "held_out_val_groups",
              "best_epoch", "best_val_loss", "best_val_macro_f1_window", "val_macro_f1_recording", "val_metric_label",
              "class_weights", "zscore_hash", "init_sha1", "n_params", "train_seconds", "checkpoint", "probs_file",
              "finished_at"]


def ckpt_path(arm, fold, seed):
    return CKPT_DIR / f"{arm}_fold{fold}_seed{seed}_{ARM_TAG[arm]}.pt"


def scratch_probs_path(arm, fold, seed):
    return PROBS_DIR / f"scratch_{arm}_fold{fold}_seed{seed}_{ARM_TAG[arm]}.npz"


def load_done():
    if not RUNS_CSV.exists():
        return {}
    df = pd.read_csv(RUNS_CSV)
    done = {}
    for r in df.itertuples():
        a, k, s = r.arm, int(r.fold), int(r.seed)
        if a in ARM_TAG and r.config_tag == ARM_TAG[a] and ckpt_path(a, k, s).exists() and \
                (a.startswith("EXT") or scratch_probs_path(a, k, s).exists()):
            done[(a, k, s)] = r
    return done


def append_run(row):
    new = not RUNS_CSV.exists()
    with open(RUNS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RUN_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**row, "finished_at": datetime.now().isoformat(timespec="seconds")})


RUN_SPECS = [(f"EXT_{f}", -1, s) for f in FEATURE_SETS for s in SEEDS] + \
            [(f"TII_{f}", k, s) for f in FEATURE_SETS for s in SEEDS for k in range(len(DAYS))]


def n_train_windows(arm, fold, seed):
    if arm.startswith("EXT"):
        return int(EXT_WMETA.recording.isin(EXT_SPLITS[seed]["train"]).sum())
    return int(TII_WMETA.mission.isin(SCRATCH_FOLDS[fold]["train"]).sum()) * N_CH


_done = load_done()
_todo = [r for r in RUN_SPECS if r not in _done]
print(f"{len(RUN_SPECS)} training runs planned | already logged with the same config tag: {len(RUN_SPECS) - len(_todo)} | "
      f"to train now: {len(_todo)}")
print("Config tags: " + ", ".join(f"{a} {t}" for a, t in ARM_TAG.items()))
_sec_per_window_epoch, RUNTIME_ESTIMATE_PRINTED = None, False
T_TRAIN_START = time.time()
for i, (arm, k, s) in enumerate(RUN_SPECS, 1):
    if (arm, k, s) in _done:
        print(f"[run {i:02d}/{len(RUN_SPECS)}] {arm} fold {k} seed {s} — already logged, skipped")
        continue
    feature = arm.split("_", 1)[1]
    tag = f"[{arm} fold {k} seed {s}]"
    if arm.startswith("EXT"):
        sp = EXT_SPLITS[s]
        itr = np.where(EXT_WMETA.recording.isin(sp["train"]).values)[0]
        iva = np.where(EXT_WMETA.recording.isin(sp["val"]).values)[0]
        train_ids = set(EXT_WMETA.recording.values[itr])
        sources = f"external .wav windows only ({len(train_ids)} train / {len(sp['val'])} val recordings); no TII data"
        hook(f"{arm} seed {s}: model trained without TII data", all(r.startswith("ext_") for r in train_ids)
             and not (train_ids & set(MISSIONS)), sources, hard=True)
        model, info = train_model(tag, s, EXT_X[feature][itr], EXT_Y[itr], EXT_X[feature][iva], EXT_Y[iva])
        _val_rec = EXT_WMETA.recording.values[iva]
        _dfv = pd.DataFrame(info["Pva"]).assign(rec=_val_rec, y=EXT_Y[iva]).groupby("rec").mean()
        val_rec_f1 = f1_score(_dfv.y.astype(int), _dfv[list(range(N_CLASSES))].values.argmax(1), average="macro",
                              labels=list(range(N_CLASSES)), zero_division=0)
        print(f"{tag} VALIDATION (within-external, held-out configurations): window macro-F1 "
              f"{info['best_val_macro_f1_window']:.4f} | recording macro-F1 {val_rec_f1:.4f} (best epoch {info['best_epoch']})")
        row = {"fold_day": "external", "n_train_recordings": len(sp["train"]), "n_val_recordings": len(sp["val"]),
               "n_test_recordings": 0, "n_train_windows": len(itr), "n_val_windows": len(iva), "n_test_windows": 0,
               "held_out_val_groups": json.dumps({CLASS_NAMES[c]: v for c, v in sp["held"].items()}),
               "val_macro_f1_recording": val_rec_f1,
               "val_metric_label": "within-external, held-out configurations", "probs_file": ""}
    else:
        fold = SCRATCH_FOLDS[k]
        X = tii_windows(feature, ASSUMED_TII_RATE, GAINS_FOR[feature][0])          # RAW (LEVEL) / NONE (SHAPE)
        sel = lambda ms: np.where(TII_WMETA.mission.isin(ms).values)[0]
        itr, iva, ite = sel(fold["train"]), sel(fold["val"]), sel(fold["test"])
        _train_days = set(TII_WMETA.day.values[itr]) | set(TII_WMETA.day.values[iva])
        sources = (f"TII days {sorted(_train_days)} only ({len(fold['train'])} train / {len(fold['val'])} val missions, "
                   f"all 5 channels as separate samples); test day {fold['day']}")
        hook(f"{arm} fold {k} seed {s}: no test-day mission in training", fold["day"] not in _train_days
             and not (set(TII_WMETA.mission.values[itr]) & set(fold["test"])), sources, hard=True)
        flat = lambda idx: X[idx].reshape(-1, N_TOKENS, N_FEAT)            # window-major, then channel
        rep = lambda idx: np.repeat(TII_WMETA["class"].values[idx].astype(np.int64), N_CH)
        model, info = train_model(tag, s, flat(itr), rep(itr), flat(iva), rep(iva))
        Pte = predict_probs(model, flat(ite), info["mu"], info["sd"]).reshape(len(ite), N_CH, N_CLASSES)
        np.savez(scratch_probs_path(arm, k, s), window_id=TII_WMETA.window_id.values[ite].astype(str),
                 mission=TII_WMETA.mission.values[ite].astype(str), probs_window=Pte.astype(np.float32))
        row = {"fold_day": fold["day"], "n_train_recordings": len(fold["train"]), "n_val_recordings": len(fold["val"]),
               "n_test_recordings": len(fold["test"]), "n_train_windows": len(itr) * N_CH, "n_val_windows": len(iva) * N_CH,
               "n_test_windows": len(ite) * N_CH, "held_out_val_groups": "", "val_macro_f1_recording": np.nan,
               "val_metric_label": "TII inner validation (training days)", "probs_file": str(scratch_probs_path(arm, k, s))}
    torch.save({"state": info["state"], "mu": info["mu"], "sd": info["sd"], "class_weights": info["class_weights"],
                "arm": arm, "fold": k, "seed": s, "config_tag": ARM_TAG[arm], "data_sources": sources,
                "history": info["history"]}, ckpt_path(arm, k, s))
    append_run({**row, "arm": arm, "fold": k, "seed": s, "config_tag": ARM_TAG[arm], "data_sources": sources,
                "best_epoch": info["best_epoch"], "best_val_loss": round(info["best_val_loss"], 6),
                "best_val_macro_f1_window": round(info["best_val_macro_f1_window"], 6),
                "class_weights": json.dumps(np.round(info["class_weights"], 4).tolist()),
                "zscore_hash": info["zscore_hash"], "init_sha1": info["init_sha1"], "n_params": info["n_params"],
                "train_seconds": round(info["train_seconds"], 1), "checkpoint": str(ckpt_path(arm, k, s))})
    print(f"[run {i:02d}/{len(RUN_SPECS)}] {arm} fold {k} seed {s} done in {info['train_seconds']:.0f} s")
    if not RUNTIME_ESTIMATE_PRINTED:
        _sec_per_window_epoch = info["train_seconds"] / (n_train_windows(arm, k, s) * EPOCHS)
        _rest = [r for r in _todo if r != (arm, k, s)]
        _est = sum(n_train_windows(*r) for r in _rest) * EPOCHS * _sec_per_window_epoch
        print("⏱" * 40)
        print(f"ESTIMATED TOTAL RUNTIME: ~{(time.time() - T_NOTEBOOK_START + _est) / 60:.0f} min "
              f"(elapsed so far {(time.time() - T_NOTEBOOK_START) / 60:.0f} min + remaining training {_est / 60:.0f} min "
              f"for {len(_rest)} runs, scaled by training-window count from this run). Evaluation adds a few minutes.")
        print("⏱" * 40)
        RUNTIME_ESTIMATE_PRINTED = True
    del model, info
    gc.collect()
    torch.cuda.empty_cache()
TRAIN_SECONDS_THIS_SESSION = time.time() - T_TRAIN_START
RUNS = pd.read_csv(RUNS_CSV)
RUNS = RUNS[RUNS.apply(lambda r: r.arm in ARM_TAG and r.config_tag == ARM_TAG[r.arm], axis=1)]
assert len(load_done()) == len(RUN_SPECS), "some runs are missing; re-run this cell"
print(f"\nAll {len(RUN_SPECS)} runs available. Training time this session: {TRAIN_SECONDS_THIS_SESSION / 60:.1f} min")
print("\nData sources of every trained model:")
for r in RUNS.sort_values(["arm", "seed", "fold"]).itertuples():
    print(f"  {r.arm:<10} fold {r.fold:>2} seed {r.seed:<4} {r.data_sources}")
_ext_rows = RUNS[RUNS.arm.str.startswith("EXT")]
if len(_ext_rows):
    print("\nEXT validation (within-external, held-out configurations):")
    print(_ext_rows[["arm", "seed", "best_epoch", "best_val_loss", "best_val_macro_f1_window", "val_macro_f1_recording",
                     "held_out_val_groups"]].to_string(index=False))

## 6. Evaluation (all 95 TII missions)
- **Zero-shot arms** are diagnostics, except the pre-registered headline:
  - EXT_LEVEL × {RAW, NORMCAL} and EXT_SHAPE;
  - each on channel c1 … c5 and on MEAN5 (window probabilities averaged over the 5 channels);
  - at the assumed rate, plus `RATE_EST` if section 3b found one.
  All are reported. None is picked as best on TII.
- **Headline (pre-registered): EXT_LEVEL + NORMCAL + MEAN5 + assumed rate.**
- **References:**
  - **TII_LEVEL / TII_SHAPE scratch** (day-out, out-of-fold, same channel aggregations);
  - **RMS-CENTROID** under RAW and NORMCAL. Feature = log of the median-over-channels steady RMS per mission;
    classifier = the nearest class centroid of the external log steady RMS.
- **Mission prediction** = argmax of the mean window probabilities. Seeds are combined by averaging window
  probabilities ("avg"). Single seeds are also reported.
- **Metrics on 95 missions, pooled and per day:**
  - accuracy, macro-F1, balanced accuracy, per-class recall, confusion matrix, share of errors that are
    adjacent-class;
  - window-level metrics as secondary;
  - mission-level bootstrap 95% CIs (2000 resamples of missions within class);
  - paired mission bootstrap for zero-shot − TII scratch.
- The 2021-12-22 fold has only 10 missions. Chance macro-F1 ≈ 0.20.

In [ ]:
# ==============================================================================
# 6a. METRIC HELPERS: mission aggregation, vectorized mission-level bootstrap (shared draws => paired comparisons)
# ==============================================================================
_MIS_POS = {m: i for i, m in enumerate(MISSIONS)}
_W_MIS = TII_WMETA.mission.map(_MIS_POS).values
_W_COUNT = np.bincount(_W_MIS, minlength=len(MISSIONS))
MIS_Y = np.array([TII[m]["class"] for m in MISSIONS])
MIS_DAY = np.array([TII[m]["day"] for m in MISSIONS])
W_Y = TII_WMETA["class"].values


def to_mission(Pw):
    """(n_windows, 5) -> (95, 5): mean window probability per mission (MISSIONS order)."""
    out = np.zeros((len(MISSIONS), N_CLASSES))
    np.add.at(out, _W_MIS, Pw)
    return out / _W_COUNT[:, None]


def select_channels(P, spec):
    """P: (n_windows, 5 channels, 5 classes)."""
    return P.mean(axis=1) if spec == "MEAN5" else P[:, int(spec[1:]) - 1]


_rng = np.random.default_rng(BOOTSTRAP_SEED)
BOOT_IDX = np.concatenate([np.where(MIS_Y == c)[0][_rng.integers(0, (MIS_Y == c).sum(), (BOOTSTRAP_RESAMPLES, (MIS_Y == c).sum()))]
                           for c in range(N_CLASSES)], axis=1)          # (B, 95): missions resampled within class
_BOOT_OFF = (np.arange(BOOTSTRAP_RESAMPLES) * N_CLASSES ** 2)[:, None]


def prf_batch(cms):
    tp = np.diagonal(cms, axis1=-2, axis2=-1).astype(float)
    cs, rs = cms.sum(-2), cms.sum(-1)
    prec = np.divide(tp, cs, out=np.zeros_like(tp), where=cs > 0)
    rec = np.divide(tp, rs, out=np.zeros_like(tp), where=rs > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cm_of(y, pred):
    return np.bincount(y * N_CLASSES + pred, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)


def boot_macro_f1(pred):
    codes = MIS_Y[BOOT_IDX] * N_CLASSES + pred[BOOT_IDX] + _BOOT_OFF
    cms = np.bincount(codes.ravel(), minlength=BOOTSTRAP_RESAMPLES * N_CLASSES ** 2).reshape(-1, N_CLASSES, N_CLASSES)
    return prf_batch(cms)[2].mean(axis=1)


def ci(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return float(lo), float(hi)


def mission_metrics(pred, Pw=None):
    cm = cm_of(MIS_Y, pred)
    _, rec, f1 = prf_batch(cm)
    err = pred != MIS_Y
    res = {"mission_accuracy": float((pred == MIS_Y).mean()), "mission_macro_f1": float(f1.mean()),
           "mission_balanced_accuracy": float(rec.mean()),
           **{f"recall_c{c}": float(rec[c]) for c in range(N_CLASSES)},
           "adjacent_error_share": float((np.abs(pred - MIS_Y)[err] == 1).mean()) if err.any() else np.nan,
           "faulty_missions_called_normal": float((pred[MIS_Y > 0] == 0).mean()),
           "confusion_matrix": json.dumps(cm.tolist())}
    res["mission_macro_f1_ci_lo"], res["mission_macro_f1_ci_hi"] = ci(boot_macro_f1(pred))
    for d in DAYS:
        sel = MIS_DAY == d
        _, _, fd = prf_batch(cm_of(MIS_Y[sel], pred[sel]))
        res[f"day_{d}_macro_f1"] = float(fd.mean())
        res[f"day_{d}_accuracy"] = float((pred[sel] == MIS_Y[sel]).mean())
    if Pw is not None:
        pw = Pw.argmax(1)
        res["window_accuracy"] = float((pw == W_Y).mean())
        res["window_macro_f1"] = float(f1_score(W_Y, pw, average="macro", labels=list(range(N_CLASSES)), zero_division=0))
    return res


def paired(pred_a, pred_b):
    d = boot_macro_f1(pred_a) - boot_macro_f1(pred_b)
    point = prf_batch(cm_of(MIS_Y, pred_a))[2].mean() - prf_batch(cm_of(MIS_Y, pred_b))[2].mean()
    return float(point), *ci(d)


print(f"Bootstrap: {BOOTSTRAP_RESAMPLES} resamples of missions within class (seed {BOOTSTRAP_SEED}); the same draws are "
      f"used for every arm, so differences are paired.")

In [ ]:
# ==============================================================================
# 6b. PREDICTIONS: zero-shot (every EXT model on every TII arm input) + TII scratch out-of-fold assembly
# ==============================================================================
def rate_tag(r):
    return "assumed16k" if r == ASSUMED_TII_RATE else f"est{r}"


def load_model(path):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = AudioMamba().to(device)
    m.load_state_dict(ck["state"])
    m.eval()
    assert verify_model(m)[0]
    return m, ck["mu"], ck["sd"]


def save_probs(path, P):
    Pm = np.stack([to_mission(select_channels(P, spec)) for spec in CHANNEL_SPECS], axis=1)   # (95, 6, 5)
    np.savez(path, window_id=TII_WMETA.window_id.values.astype(str), mission=TII_WMETA.mission.values.astype(str),
             probs_window=P.astype(np.float32), mission_names=np.array(MISSIONS), channel_specs=np.array(CHANNEL_SPECS),
             probs_mission=Pm.astype(np.float32))


PRED = {}            # (model, gain, rate, seed) -> (n_windows, 5 channels, 5 classes)
_t_eval = time.time()
for f in FEATURE_SETS:
    arm = f"EXT_{f}"
    for s in SEEDS:
        model = None
        for r in RATE_VARIANTS:
            for g in GAINS_FOR[f]:
                path = PROBS_DIR / f"zeroshot_{arm}_seed{s}_{g}_{rate_tag(r)}_{ARM_TAG[arm]}.npz"
                if path.exists():
                    PRED[(arm, g, r, s)] = np.load(path)["probs_window"]
                    continue
                if model is None:
                    model, mu, sd = load_model(ckpt_path(arm, -1, s))
                X = tii_windows(f, r, g)
                P = predict_probs(model, X.reshape(-1, N_TOKENS, N_FEAT), mu, sd).reshape(len(X), N_CH, N_CLASSES)
                save_probs(path, P)
                PRED[(arm, g, r, s)] = P
                print(f"  zero-shot {arm} seed {s} {g} {rate_tag(r)}: {len(X)} windows x {N_CH} channels")
        del model
        torch.cuda.empty_cache()

_wid_pos = {w: i for i, w in enumerate(TII_WMETA.window_id)}
for f in FEATURE_SETS:
    arm = f"TII_{f}"
    for s in SEEDS:
        P = np.full((len(TII_WMETA), N_CH, N_CLASSES), np.nan, np.float32)
        for k in range(len(DAYS)):
            z = np.load(scratch_probs_path(arm, k, s))
            P[[_wid_pos[w] for w in z["window_id"]]] = z["probs_window"]
        assert np.isfinite(P).all(), f"{arm} seed {s}: some windows have no out-of-fold prediction"
        save_probs(PROBS_DIR / f"scratch_{arm}_seed{s}_pooled_{ARM_TAG[arm]}.npz", P)
        PRED[(arm, GAINS_FOR[f][0], ASSUMED_TII_RATE, s)] = P

for key in list(PRED):                          # seed average of window probabilities
    model_, g, r, s = key
    if s == SEEDS[0]:
        PRED[(model_, g, r, "avg")] = np.mean([PRED[(model_, g, r, ss)] for ss in SEEDS], axis=0)
print(f"Predictions ready: {len(PRED)} (model, gain, rate, seed) combinations | {time.time() - _t_eval:.0f} s")

In [ ]:
# ==============================================================================
# 6c. ALL ARMS -> audio_zeroshot_eval.csv (one row per arm/seed) + the main table
# ==============================================================================
_rows, MIS_PRED = [], {}


def arm_name(model_, g, spec, r):
    return f"{model_}|{g}|{spec}|{rate_tag(r)}"


for (model_, g, r, s), P in PRED.items():
    for spec in CHANNEL_SPECS:
        Pw = select_channels(P, spec)
        pred = to_mission(Pw).argmax(1)
        name = arm_name(model_, g, spec, r)
        MIS_PRED[(name, s)] = pred
        is_head = (model_ == HEADLINE["model"] and g == HEADLINE["gain"] and spec == HEADLINE["channels"]
                   and r == HEADLINE["rate"])
        role = "HEADLINE" if is_head else ("reference (in-domain)" if model_.startswith("TII") else "diagnostic (zero-shot)")
        _rows.append({"arm": name, "role": role, "model": model_, "feature": model_.split("_", 1)[1], "gain": g,
                      "channels": spec, "rate_hz": r, "seed": s, **mission_metrics(pred, Pw)})

# ---- RMS-CENTROID baseline (no training) ----
_ext_log = np.log(EXT_META.steady_rms.values)
CENTROIDS = np.array([_ext_log[EXT_META["class"].values == c].mean() for c in range(N_CLASSES)])
_steady = TII_META.loc[MISSIONS, _steady_cols].values                         # (95, 5)
_scale = np.stack([EXT_NORMAL_RMS / NORMCAL_G[TII[m]["day"]] for m in MISSIONS])   # amplitude scale per channel
for g, v in (("RAW", np.median(_steady, axis=1)), ("NORMCAL", np.median(_steady * _scale, axis=1))):
    pred = np.abs(np.log(v)[:, None] - CENTROIDS[None, :]).argmin(1)
    name = f"RMS_CENTROID|{g}|median5|n/a"
    MIS_PRED[(name, "n/a")] = pred
    _rows.append({"arm": name, "role": "baseline (no training)", "model": "RMS_CENTROID", "feature": "log steady RMS",
                  "gain": g, "channels": "median over 5", "rate_hz": np.nan, "seed": "n/a", **mission_metrics(pred)})
print("External class centroids of log steady RMS: " + ", ".join(f"{CLASS_NAMES[c]} {np.exp(CENTROIDS[c]):.0f}"
                                                                    for c in range(N_CLASSES)))

EVAL = pd.DataFrame(_rows)
# ---- paired deltas (seed-averaged): zero-shot minus TII scratch with the same feature set and channel aggregation ----
EVAL["delta_vs_TII_scratch"], EVAL["delta_ci_lo"], EVAL["delta_ci_hi"] = np.nan, np.nan, np.nan
for i, row in EVAL.iterrows():
    if row["model"].startswith("EXT") and row["seed"] == "avg":
        ref = arm_name(f"TII_{row['feature']}", GAINS_FOR[row["feature"]][0], row["channels"], ASSUMED_TII_RATE)
        d, lo, hi = paired(MIS_PRED[(row["arm"], "avg")], MIS_PRED[(ref, "avg")])
        EVAL.loc[i, ["delta_vs_TII_scratch", "delta_ci_lo", "delta_ci_hi"]] = d, lo, hi
EVAL.to_csv(EVAL_CSV, index=False)
print(f"Saved {EVAL_CSV} ({len(EVAL)} rows: every arm x seed, plus seed-averaged 'avg' and the baselines)")

HEAD_ARM = arm_name(HEADLINE["model"], HEADLINE["gain"], HEADLINE["channels"], HEADLINE["rate"])
REF_ARM = arm_name("TII_LEVEL", "RAW", "MEAN5", ASSUMED_TII_RATE)
_fmt = lambda r: f"{r['mission_macro_f1']:.3f} [{r['mission_macro_f1_ci_lo']:.3f}, {r['mission_macro_f1_ci_hi']:.3f}]"
_avg = EVAL[(EVAL.seed == "avg") | (EVAL.seed == "n/a")]
_main = _avg[(_avg.channels.isin(["MEAN5", "median over 5"]))].copy()
_main["mission macro-F1 [95% CI]"] = _main.apply(_fmt, axis=1)
_main["Δ vs TII scratch [95% CI]"] = _main.apply(
    lambda r: f"{r['delta_vs_TII_scratch']:+.3f} [{r['delta_ci_lo']:+.3f}, {r['delta_ci_hi']:+.3f}]"
    if np.isfinite(r["delta_vs_TII_scratch"]) else "", axis=1)
_main = _main.sort_values(["role", "arm"], key=lambda c: c.map({"HEADLINE": 0, "reference (in-domain)": 1,
                                                                "baseline (no training)": 2, "diagnostic (zero-shot)": 3})
                          if c.name == "role" else c)
print("\n" + "=" * 150)
print("MAIN TABLE — 95 TII missions, mission level, seed-averaged probabilities (MEAN5 / median over channels)")
print("=" * 150)
print(_main[["role", "arm", "mission macro-F1 [95% CI]", "mission_accuracy", "mission_balanced_accuracy",
             "adjacent_error_share", "window_macro_f1", "Δ vs TII scratch [95% CI]"]].to_string(index=False,
                                                                                               float_format=lambda v: f"{v:.3f}"))
print(f"Chance macro-F1 ≈ {CHANCE_MACRO_F1:.2f}.")

In [ ]:
# ==============================================================================
# 6d. PER-CHANNEL + PER-DAY TABLES, PER-CLASS RECALL, CONFUSION MATRICES, CONTEXT
# ==============================================================================
_pc = _avg[_avg.model != "RMS_CENTROID"].copy()
_pc["col"] = _pc.model + " " + _pc.gain + " " + _pc.rate_hz.map(lambda r: rate_tag(int(r)))
PER_CHANNEL = _pc.pivot(index="channels", columns="col", values="mission_macro_f1").reindex(CHANNEL_SPECS)
print("Per channel — mission macro-F1 (seed-averaged). Diagnostics only: no channel is selected on TII.")
print(PER_CHANNEL.round(3).to_string())

_per_day = []
for name in [HEAD_ARM, REF_ARM, "RMS_CENTROID|NORMCAL|median5|n/a", "RMS_CENTROID|RAW|median5|n/a"]:
    r = EVAL[(EVAL.arm == name) & EVAL.seed.isin(["avg", "n/a"])]
    if len(r):
        r = r.iloc[0]
        _per_day.append({"arm": name, **{f"{d}{' (SMALL, 10)' if d == SMALL_FOLD_DAY else ''}":
                                         f"{r[f'day_{d}_macro_f1']:.3f} / {r[f'day_{d}_accuracy']:.3f}" for d in DAYS}})
print("\nPer day — mission macro-F1 / accuracy (seed-averaged). Each day has only 2–8 missions per class:")
print(pd.DataFrame(_per_day).set_index("arm").to_string())

print("\nPer-class mission recall (seed-averaged):")
_rec_cols = [f"recall_c{c}" for c in range(N_CLASSES)]
print(_main[["arm"] + _rec_cols + ["faulty_missions_called_normal"]].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

print("\nPer seed (headline and reference):")
print(EVAL[EVAL.arm.isin([HEAD_ARM, REF_ARM])][["arm", "seed", "mission_macro_f1", "mission_macro_f1_ci_lo",
                                                 "mission_macro_f1_ci_hi", "window_macro_f1"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))


def plot_cm(ax, cm, title):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.grid(False)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{int(cm[i, j])}", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xlabel("predicted (broken propellers)")
    ax.set_ylabel("true")
    ax.set_title(title, fontsize=9)


_cm_arms = [(HEAD_ARM, "avg", "HEADLINE zero-shot\nEXT_LEVEL + NORMCAL + MEAN5"),
            (REF_ARM, "avg", "TII scratch (day-out)\nLEVEL + MEAN5"),
            ("RMS_CENTROID|NORMCAL|median5|n/a", "n/a", "RMS-CENTROID\nNORMCAL")]
fig, axs = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (name, s, title) in zip(axs, _cm_arms):
    plot_cm(ax, cm_of(MIS_Y, MIS_PRED[(name, s)]), title)
save_fig(fig, "6d_confusion_matrices",
         "Mission level, 95 TII missions, seed-averaged probabilities. NORMCAL uses other-day TII Normal loudness "
         "(semi-supervised). TII inputs use the assumed 16 kHz rate.")

fig, ax = plt.subplots(figsize=(11, 4))
_cols = list(PER_CHANNEL.columns)
_wd = 0.8 / len(_cols)
for j, col in enumerate(_cols):
    ax.bar(np.arange(len(CHANNEL_SPECS)) + j * _wd, PER_CHANNEL[col].values, width=_wd, label=col)
ax.axhline(CHANCE_MACRO_F1, color="#52514e", ls="--", lw=1)
ax.set_xticks(np.arange(len(CHANNEL_SPECS)) + 0.4 - _wd / 2)
ax.set_xticklabels(CHANNEL_SPECS)
ax.set_ylabel("mission macro-F1")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, ncol=2)
ax.set_title("Per channel: zero-shot arms vs TII scratch")
save_fig(fig, "6d_per_channel",
         "Seed-averaged mission macro-F1 on 95 TII missions. Dashed line = chance (~0.20). Diagnostics only: no arm is "
         "chosen on TII. " + RATE_CAVEAT)

print("\nCONTEXT ONLY — earlier runs, different front end, not for paired comparison:")
for line in CONTEXT_LINES:
    print("  " + line)

## 7. Tiers and gate (is the pretraining notebook worth running?)
These rules apply to the **headline arm's mission-level macro-F1** (seed-averaged). They were fixed before any result.
- **NOT COMPATIBLE:** the 95% CI lower bound ≤ 0.30.
- **PARTIAL TRANSFER:** the lower bound > 0.30 **and** the point estimate < TII scratch − 0.05.
- **STRONG TRANSFER:** the point estimate is within 0.05 of TII scratch, or higher.

**Gate:** the next notebook (pretraining) is worth running if the tier is PARTIAL or STRONG.

**Loudness check:** does the neural zero-shot clearly beat RMS-CENTROID (NORMCAL)? "Clearly" means the paired
bootstrap 95% CI of the difference lies above 0. If not, the transfer may be **loudness-only**.

In [ ]:
# ==============================================================================
# 7. TIER VERDICT + GATE (pre-registered rules applied to the headline arm)
# ==============================================================================
_h = EVAL[(EVAL.arm == HEAD_ARM) & (EVAL.seed == "avg")].iloc[0]
_ref = EVAL[(EVAL.arm == REF_ARM) & (EVAL.seed == "avg")].iloc[0]
H_F1, H_LO, H_HI = _h.mission_macro_f1, _h.mission_macro_f1_ci_lo, _h.mission_macro_f1_ci_hi
REF_F1 = _ref.mission_macro_f1
if H_LO <= TIER_LB_NOT_COMPATIBLE:
    TIER = "NOT COMPATIBLE"
elif H_F1 < REF_F1 - TIER_MARGIN:
    TIER = "PARTIAL TRANSFER"
else:
    TIER = "STRONG TRANSFER"
GATE_PASSES = TIER in ("PARTIAL TRANSFER", "STRONG TRANSFER")
_dc, _dclo, _dchi = paired(MIS_PRED[(HEAD_ARM, "avg")], MIS_PRED[("RMS_CENTROID|NORMCAL|median5|n/a", "n/a")])
BEATS_CENTROID = _dclo > 0
_dr, _drlo, _drhi = paired(MIS_PRED[(HEAD_ARM, "avg")], MIS_PRED[(REF_ARM, "avg")])

VERDICT = [
    "=" * 110,
    "VERDICT (pre-registered rules, headline arm EXT_LEVEL + NORMCAL + MEAN5 + assumed 16 kHz, seed-averaged)",
    "=" * 110,
    f"Headline mission macro-F1 {H_F1:.3f} [95% CI {H_LO:.3f}, {H_HI:.3f}] on 95 TII missions (chance ~{CHANCE_MACRO_F1:.2f}).",
    f"TII scratch (LEVEL, MEAN5, day-out) {REF_F1:.3f} [{_ref.mission_macro_f1_ci_lo:.3f}, {_ref.mission_macro_f1_ci_hi:.3f}]; "
    f"paired difference headline − scratch {_dr:+.3f} [{_drlo:+.3f}, {_drhi:+.3f}].",
    f"TIER: {TIER}",
    f"GATE: {'PASSES — the pretraining notebook is worth running.' if GATE_PASSES else 'DOES NOT PASS — do not run the pretraining notebook on this basis.'}",
    f"Neural zero-shot vs RMS-CENTROID (NORMCAL): paired difference {_dc:+.3f} [{_dclo:+.3f}, {_dchi:+.3f}] -> "
    + ("the neural model CLEARLY beats the loudness-only baseline." if BEATS_CENTROID else
       "the neural model does NOT clearly beat the loudness-only baseline: the transfer may be LOUDNESS-ONLY."),
    "Caveats: NORMCAL is a semi-supervised gain calibration (TII Normal loudness from other days); the TII sample rate "
    f"is assumed ({ASSUMED_TII_RATE} Hz; scan result: {RATE_EST}); this tests compatibility, not whether pretraining "
    "improves final TII results.",
]
print("\n".join(VERDICT))

In [ ]:
# ==============================================================================
# FINAL: hook table, deliverables summary, runtime, SAVE REMINDER
# ==============================================================================
print("=" * 110)
print("VERIFICATION HOOKS")
print("=" * 110)
for h in HOOKS:
    print(f"  {'PASS' if h['ok'] else ('FAIL' if h['hard'] else 'FLAG')}  [{'hard' if h['hard'] else 'soft'}] {h['hook']}")
_flags = [h for h in HOOKS if not h["ok"]]
print(f"{len(HOOKS)} hooks: {sum(h['ok'] for h in HOOKS)} PASS, {len(_flags)} FLAG (soft deviations, listed above)")

SUMMARY = "\n".join(COMPAT_LINES + [""] + VERDICT + [
    "",
    f"Runtime: total notebook {(time.time() - T_NOTEBOOK_START) / 60:.1f} min this session (training "
    f"{TRAIN_SECONDS_THIS_SESSION / 60:.1f} min; runs already logged were skipped).",
    f"Outputs in {OUT}: external_metadata.csv, tii_audio_missions.csv, compat_rate_scan.csv, compat_overlap_screen.csv, "
    f"{RUNS_CSV.name}, {EVAL_CSV.name}, {PROBS_DIR.name}/, {CKPT_DIR.name}/, figures/ ({len(FIGURES)} figures).",
])
(OUT / "audio_zeroshot_summary.txt").write_text(SUMMARY, encoding="utf-8")
print("\n" + SUMMARY)

print("\n" + "💾" * 50)
print("💾  DONE. To keep /kaggle/working (CSVs, probabilities, checkpoints, figures), click")
print("💾      \"Save Version\"  ->  \"Save & Run All (Commit)\"")
print("💾  Otherwise everything in /kaggle/working is LOST when this session ends.")
print("💾" * 50)